# Conformal uncertainty for digital-rock property prediction: full experiment 

**How to run.** Runtime → Change runtime type → L4 GPU (High-RAM if available). Then Runtime → Run all.
Every expensive stage is cached to Google Drive, so after a disconnect just run all again and it resumes.

**Profiles** (set `PROFILE` in the next cell):

| Profile | Data | Seeds | Purpose |
|---|---|---|---|
| `paper` | all valid DeePore volumes | 10 | results for the manuscript |
| `standard` | 8,000 volumes | 5 | a faster full pass |
| `smoke` | small **synthetic** stand-in | 2 | code test only; figures are watermarked and must never be reported |

The notebook measures training speed on your GPU after the first run and prints a projected total time.

In [ ]:
# ============================== 1. Setup ==============================
import os, sys, json, time, math, hashlib, shutil, subprocess, platform, warnings, itertools, glob, zipfile
from dataclasses import dataclass, asdict, field
from math import ceil

PROFILE = os.environ.get("ROCK_PROFILE", "paper")      # "paper" | "standard" | "smoke"
assert PROFILE in ("paper", "standard", "smoke")

import numpy as np, pandas as pd, scipy, scipy.stats as st, sklearn
import torch, torch.nn as nn, torch.nn.functional as F
import h5py
import matplotlib, matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    from IPython.display import display, Markdown
except Exception:                                      # plain-python fallback
    display = print
    Markdown = lambda s: s

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE == "cuda" else 0.0
print(f"profile={PROFILE} | device={DEVICE} ({GPU_NAME}, {VRAM_GB:.1f} GB) | torch {torch.__version__} | cpu cores {os.cpu_count()}")
if PROFILE != "smoke" and DEVICE != "cuda":
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> L4 GPU, then run all again.")
torch.backends.cudnn.benchmark = True
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True

In [ ]:
# ============================== 2. Configuration ==============================
@dataclass
class CFG:
    profile: str = PROFILE
    # ---- data ----
    n_max: int = 0                 # 0 = all valid volumes; otherwise a seeded uniform subsample
    work_res: int = 64             # working resolution of the network input
    hi_res: int = 128              # high-resolution variant (subset only, resolution study)
    poremap_cap: int = 24          # cap (px, full-res) of the erosion-distance pore-size map
    res_subset_n: int = 2000       # subset used for the resolution / representation study
    conn_subset_n: int = 300       # subset used for the connectivity check
    data_seed: int = 12345
    # ---- splits ----
    frac_train: float = 0.55
    frac_val: float = 0.05         # monitoring only; never used for calibration or testing
    frac_cal: float = 0.20
    frac_test: float = 0.20
    # ---- model / optimisation ----
    base: int = 16
    p_drop: float = 0.10
    epochs: int = 24
    batch: int = 48
    lr: float = 2e-3
    wd: float = 1e-4
    w_nll: float = 0.5
    w_q: float = 1.0
    # ---- experiment sizes ----
    n_seeds: int = 10
    n_abl: int = 3                 # seeds for the mean-only / dedicated-Gaussian ablation
    ens_M: int = 5                 # deep-ensemble members (split 0)
    mc_T: int = 30                 # MC-dropout passes
    mc_T_grid: tuple = (5, 10, 20, 30, 50, 100)
    # ---- conformal ----
    alpha: float = 0.10
    q_lo: tuple = (0.01, 0.025, 0.05, 0.10, 0.15)       # lower quantile levels; upper = 1 - lower
    sweep_targets: tuple = (0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.98)
    eps_rel: float = 1e-3          # epsilon of the normalized score, relative to the training sd of the target
    n_groups: int = 5              # Mondrian groups
    # ---- shift experiments ----
    ood_top_frac: float = 0.25
    n_families: int = 10
    n_family_folds: int = 3
    # ---- resampling sizes for cheap (CPU) analyses ----
    n_repart: int = 300
    n_tilt_rep: int = 20
    # ---- triage ----
    tol_factors: tuple = (1.25, 1.5, 2.0, 3.0)          # tolerance as a multiplicative factor on permeability
    risk_levels: tuple = (0.01, 0.05, 0.10)
    ltt_delta: float = 0.10
    guard_delta: float = 0.05
    # ---- units ----
    voxel_um: float = 3.0          # illustrative voxel size (um) used ONLY for the mD conversion example
    # ---- stage toggles ----
    run_ablations: bool = True
    run_shift: bool = True
    run_resolution: bool = True
    run_hi_res: bool = True

cfg = CFG()
if PROFILE == "standard":
    cfg.n_max, cfg.n_seeds, cfg.epochs, cfg.n_abl, cfg.ens_M = 8000, 5, 20, 2, 3
    cfg.res_subset_n, cfg.run_hi_res = 1500, False
if PROFILE == "smoke":
    cfg.n_max, cfg.work_res, cfg.hi_res, cfg.poremap_cap = 0, 16, 32, 8
    cfg.res_subset_n, cfg.conn_subset_n = 200, 16
    cfg.base, cfg.epochs, cfg.batch = 8, 4, 32
    cfg.n_seeds, cfg.n_abl, cfg.ens_M, cfg.mc_T, cfg.mc_T_grid = 3, 1, 2, 6, (2, 4, 6, 8)
    cfg.n_families, cfg.n_family_folds, cfg.n_repart, cfg.n_tilt_rep = 6, 2, 60, 4

Q_LEVELS = list(cfg.q_lo) + [1 - q for q in cfg.q_lo]         # 10 quantile heads
N_Q = len(Q_LEVELS)

# Targets: (name, DeePore column, transform, axis label). Columns follow DeePore's VarNames.txt (0-based).
PROPS = [
    ("log_k",   0,  "log10", r"$\log_{10}k$  [px$^2$]"),
    ("log_F",   1,  "log10", r"$\log_{10}F$  (formation factor)"),
    ("tau",     4,  "id",    r"tortuosity $\tau$"),
    ("log_ssa", 10, "log10", r"$\log_{10}$ specific surface  [px$^{-1}$]"),
    ("log_rp",  7,  "log10", r"$\log_{10}$ mean pore radius  [px]"),
    ("E_rel",   14, "id",    r"relative Young's modulus"),
]
PNAMES = [p[0] for p in PROPS]; P = len(PROPS); PLABEL = {p[0]: p[3] for p in PROPS}
PSHORT = {"log_k": "permeability", "log_F": "formation factor", "tau": "tortuosity",
          "log_ssa": "specific surface", "log_rp": "pore radius", "E_rel": "Young's modulus"}
PAXIS = {"log_k": r"$\log_{10}k$ [px$^2$]", "log_F": r"$\log_{10}F$", "tau": r"$\tau$", "log_ssa": r"$\log_{10}S$ [px$^{-1}$]",
         "log_rp": r"$\log_{10}r_p$ [px]", "E_rel": r"$E/E_s$"}
J_K = 0                                                       # index of the primary property (log10 k)

# ---- paths ----
IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
DRIVE_OK = False
if IN_COLAB and PROFILE != "smoke":
    try:
        from google.colab import drive
        drive.mount("/content/drive"); DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
    except Exception as e:
        print("Drive not mounted (", e, "). Outputs stay on the runtime disk and are lost on disconnect.")
BASE = os.environ.get("ROCK_BASE", "/content" if IN_COLAB else os.getcwd())
ROOT = os.path.join("/content/drive/MyDrive" if DRIVE_OK else BASE, "conformal_rock_v2", PROFILE)   # durable outputs
LOCAL = os.path.join(BASE, "rock_local", PROFILE)                                                     # fast scratch
D = {k: os.path.join(ROOT, k) for k in ["ckpt", "preds", "tables", "figures", "splits", "logs"]}
D.update(raw=os.path.join(LOCAL, "raw"), proc=os.path.join(LOCAL, "proc"), proc_cache=os.path.join(ROOT, "proc_cache"))
for k, v in D.items():
    if k != "proc_cache": os.makedirs(v, exist_ok=True)
print("durable outputs:", ROOT); print("scratch        :", LOCAL)

TIMERS = {}
def tic(name): TIMERS[name] = -time.time()
def toc(name):
    TIMERS[name] += time.time(); print(f"[time] {name}: {TIMERS[name]/60:.1f} min"); return TIMERS[name]
def save_json(obj, path):
    with open(path, "w") as f: json.dump(obj, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
def load_json(path):
    with open(path) as f: return json.load(f)
def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(s)
def save_table(df, name, show=True, floatfmt=4):
    """Every number that goes into the paper is written here as CSV, so text, tables and figures share one source."""
    df.to_csv(os.path.join(D["tables"], name + ".csv"), index=False)
    if show: display(df.round(floatfmt))
    return df
save_json(asdict(cfg), os.path.join(D["logs"], "config.json"))

In [ ]:
# ============================== 3. Figure style ==============================
# Fixed categorical order (colour-blind validated on adjacent pairs); line style and marker shape repeat the
# identity so nothing depends on colour alone. Conformal methods: solid lines, filled markers.
# Heuristic baselines: dashed lines, open markers.
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#ffffff"
METHODS = {   # key: (label, colour, marker, linestyle, is_conformal)
    "split":    ("Split conformal",       "#2a78d6", "o", "-",  True),
    "norm":     ("Normalized conformal",  "#eb6834", "s", "-",  True),
    "cqr":      ("CQR",                   "#1baf7a", "^", "-",  True),
    "mondrian": ("Mondrian conformal",    "#eda100", "D", "-",  True),
    "gauss":    ("Gaussian head",         "#e87ba4", "v", "--", False),
    "mcd":      ("MC-dropout (means)",    "#008300", "P", "--", False),
    "mcd_tot":  ("MC-dropout (total)",    "#6250d6", "X", "--", False),
    "ens":      ("Deep ensemble",         "#e34948", "*", "--", False),
}
CONF = [m for m, v in METHODS.items() if v[4]]; HEUR = [m for m, v in METHODS.items() if not v[4]]
MLAB = {m: v[0] for m, v in METHODS.items()}; MCOL = {m: v[1] for m, v in METHODS.items()}
SEQ = matplotlib.colors.LinearSegmentedColormap.from_list("seqblue", ["#eef5fd", "#b7d3f6", "#5598e7", "#1c5cab", "#0d366b"])
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 1.6, "lines.markersize": 5, "legend.frameon": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "mathtext.default": "regular",
})
def mstyle(m, line=True):
    lab, col, mk, ls, conf = METHODS[m]
    d = dict(color=col, marker=mk, markersize=5.5, markerfacecolor=col if conf else SURF, markeredgecolor=col, markeredgewidth=1.2)
    d["linestyle"] = ls if line else "none"
    return d
def panel_letters(axes, x=-0.02, y=1.08):
    for i, ax in enumerate(np.ravel(axes)):
        ax.text(x, y, "(" + "abcdefghijkl"[i] + ")", transform=ax.transAxes, fontweight="bold", ha="right", va="bottom")
def savefig(fig, name):
    if PROFILE == "smoke":      # synthetic data: make it impossible to mistake for a result
        fig.text(0.5, 0.5, "SYNTHETIC SMOKE TEST\nnot a result", color="#e34948", alpha=0.22, fontsize=30,
                 ha="center", va="center", rotation=20, fontweight="bold", zorder=100)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(D["figures"], f"{name}.{ext}"), bbox_inches="tight")
    plt.show(); plt.close(fig)
def prop_ticks(ax, props=None):
    props = PNAMES if props is None else props
    ax.set_xticks(range(len(props))); ax.set_xticklabels([PSHORT[p] for p in props], rotation=30, ha="right", rotation_mode="anchor", fontsize=7.5)
def method_legend(fig_or_ax, methods, **kw):
    hs = [matplotlib.lines.Line2D([], [], label=MLAB[m], **mstyle(m)) for m in methods]
    return fig_or_ax.legend(handles=hs, **kw)

## 4. Data: rebuild from the public DeePore file

**Source.** `DeePore_Dataset.h5` from Zenodo record 4297035 (Rabbani et al., 2020): 17,700 binary volumes of 256³ voxels in dataset `X`
and 1,515 pore-network-derived features per volume in dataset `Y`. The first 15 features are the single-value properties
(permeability in px², formation factor, tortuosity, ...). Lengths are in voxels of the 256³ grid.

**Exclusions.** We apply DeePore's own outlier rule (`prep()` in the authors' code): drop a sample if any feature is NaN or infinite,
or formation factor > 120, tortuosity > 1.8, permeability < 1e-4 px², cementation factor < 1e-5, relative Young's modulus > 0.7.
The count removed by each rule is reported.

**Input representation (answers Reviewer 2, comment 1).** The labels belong to the 256³ geometry, so the input must stay a faithful
description of that same geometry and field of view. We therefore **do not crop and do not re-binarise**. Each volume is coarse-grained
to the working resolution with two channels computed at full resolution on the GPU:

1. **Pore fraction**: block mean of the binary pore indicator. Porosity is conserved exactly.
2. **Local pore size**: block maximum of an erosion-distance map (octagonal metric, capped), i.e. the largest local distance to solid
   inside the block. This keeps the sub-voxel pore-size information that controls hydraulic conductance and that plain down-sampling destroys.

Section 8 quantifies what this buys: accuracy for binarised input, pore-fraction only, half resolution and double resolution, on the same split.
No porosity "label" is used anywhere: DeePore does not ship one, and porosity is an exact voxel count (Section 8).

In [ ]:
# ============================== 4a. Download and verify ==============================
ZENODO_URL = "https://zenodo.org/records/4297035/files/DeePore_Dataset.h5?download=1"
ZENODO_MD5 = "d56fbd79f451353af131402c42b179d7"        # as listed on the Zenodo record
H5_OVERRIDE = os.environ.get("ROCK_H5", "")             # set to an existing copy (e.g. on Drive) to skip the download
H5_PATH = H5_OVERRIDE or os.path.join(D["raw"], "SYNTHETIC_smoke.h5" if PROFILE == "smoke" else "DeePore_Dataset.h5")
PROC_META = os.path.join(D["proc"], "proc_meta.json")

def restore_proc_cache():
    """If the processed arrays were cached on Drive in an earlier session, copy them back to fast local disk."""
    if os.path.exists(PROC_META) or not os.path.isdir(D["proc_cache"]): return
    if os.path.exists(os.path.join(D["proc_cache"], "proc_meta.json")):
        print("restoring processed data from Drive cache ..."); shutil.copytree(D["proc_cache"], D["proc"], dirs_exist_ok=True)
restore_proc_cache()

def make_synthetic_h5(path, n=900, Dv=64, seed=0):
    """SMOKE PROFILE ONLY. Writes a small file in DeePore's layout (X: n x D^3 uint8 with 1 = solid, Y: n x 1515 x 1)
    from thresholded Gaussian random fields with made-up labels. It exists to test the code path and nothing else."""
    rng = np.random.default_rng(seed)
    fr = np.fft.fftfreq(Dv); k2 = fr[:, None, None]**2 + fr[None, :, None]**2 + fr[None, None, :]**2
    Ls = [1.5, 2.0, 2.5, 3.0, 3.5, 4.5, 2.2, 3.2, 4.0, 2.8]
    with h5py.File(path, "w") as f:
        X = f.create_dataset("X", (n, Dv, Dv, Dv), dtype="uint8", chunks=(1, Dv, Dv, Dv), compression="gzip", compression_opts=1)
        Y = f.create_dataset("Y", (n, 1515, 1), dtype="float64")
        for i in range(n):
            L = Ls[(i * 10) // n] * rng.uniform(0.9, 1.1)              # "families" contiguous in index
            phi = rng.uniform(0.12, 0.45)
            fld = np.fft.ifftn(np.fft.fftn(rng.standard_normal((Dv, Dv, Dv))) * np.exp(-0.5 * k2 * (2 * np.pi * L)**2)).real
            pore = fld < np.quantile(fld, phi)
            faces = (pore[1:] != pore[:-1]).sum() + (pore[:, 1:] != pore[:, :-1]).sum() + (pore[:, :, 1:] != pore[:, :, :-1]).sum()
            sv = faces / Dv**3
            hetero = 0.05 + 0.25 * (1 - phi)                           # noisier labels for tight samples
            y = np.full(1515, 0.5)
            y[0] = max(0.2 * phi**3 / sv**2 * 10**rng.normal(0, hetero), 2e-4)
            y[1] = min(phi**(-rng.uniform(1.6, 2.0)), 119.0); y[2] = 1.8; y[3] = 1e-3
            y[4] = min(1 + 0.5 * (1 - phi) * rng.uniform(0.8, 1.2), 1.79); y[5] = 4; y[6] = L / 2
            y[7] = L * (0.8 + phi) * 10**rng.normal(0, 0.03); y[8] = 3 * L; y[9] = 2
            y[10] = sv * 10**rng.normal(0, 0.02); y[11] = 0.8; y[12] = 0.8; y[13] = 2 * L
            y[14] = min(0.6 * (1 - phi)**2 * rng.uniform(0.9, 1.0), 0.69)
            if i % 97 == 5: y[0] = 1e-6                                # a few rule violations to exercise the filter
            if i % 211 == 7: y[20] = np.nan
            X[i] = (~pore).astype("uint8"); Y[i, :, 0] = y

if not os.path.exists(PROC_META):
    if PROFILE == "smoke":
        if not os.path.exists(H5_PATH): make_synthetic_h5(H5_PATH)
        DATA_SOURCE = "SYNTHETIC stand-in (smoke profile)"; md5 = "n/a"
    else:
        if not os.path.exists(H5_PATH):
            tic("download")
            subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", H5_PATH + ".part", ZENODO_URL], check=True)
            os.replace(H5_PATH + ".part", H5_PATH); toc("download")
        h = hashlib.md5()
        with open(H5_PATH, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 24), b""): h.update(chunk)
        md5 = h.hexdigest(); print("md5:", md5, "(matches Zenodo)" if md5 == ZENODO_MD5 else "(DOES NOT MATCH ZENODO)")
        if md5 != ZENODO_MD5 and not H5_OVERRIDE:
            raise SystemExit("Checksum mismatch: the download is incomplete or the file changed. Delete it and run again.")
        DATA_SOURCE = f"DeePore_Dataset.h5, Zenodo 4297035, md5 {md5}"
    with h5py.File(H5_PATH, "r") as f:
        print("datasets:", {k: (f[k].shape, str(f[k].dtype), f[k].chunks, f[k].compression) for k in f.keys()})
        assert "X" in f and "Y" in f, "expected datasets 'X' and 'Y' as in the DeePore reader"
        N_ALL = f["X"].shape[0]
        FULL_RES = int([s for s in f["X"].shape[1:] if s > 1][0])
        Yall = np.squeeze(np.asarray(f["Y"][:, :, ...], dtype=np.float64))          # (N_ALL, 1515)
    print(f"{N_ALL} volumes of {FULL_RES}^3; label array {Yall.shape}")

In [ ]:
# ============================== 4b. Exclusions, targets, sample selection ==============================
if not os.path.exists(PROC_META):
    y15 = Yall[:, :15]
    rules = {
        "NaN or Inf in any feature":            ~np.isfinite(Yall).all(1),
        "formation factor > 120":               y15[:, 1] > 120,
        "tortuosity > 1.8":                     y15[:, 4] > 1.8,
        "permeability < 1e-4 px^2":             y15[:, 0] < 1e-4,
        "cementation factor < 1e-5":            y15[:, 2] < 1e-5,
        "relative Young's modulus > 0.7":       y15[:, 14] > 0.7,
    }
    extra = np.zeros(N_ALL, bool)                                  # targets we log-transform must be positive
    for name, col, tr, _ in PROPS:
        if tr == "log10": extra |= ~(y15[:, col] > 0)
    rules["non-positive value in a log-transformed target"] = extra
    bad = np.zeros(N_ALL, bool); rows = []
    for k, m in rules.items():
        m = np.asarray(m, bool)
        rows.append(dict(rule=k, n_flagged=int(m.sum()), n_new=int((m & ~bad).sum()))); bad |= m
    rows.append(dict(rule="TOTAL excluded", n_flagged=int(bad.sum()), n_new=int(bad.sum())))
    rows.append(dict(rule="valid volumes", n_flagged=int((~bad).sum()), n_new=int((~bad).sum())))
    excl_df = pd.DataFrame(rows)
    valid = np.where(~bad)[0]
    rng = np.random.default_rng(cfg.data_seed)
    sel = valid if (cfg.n_max == 0 or cfg.n_max >= len(valid)) else np.sort(rng.choice(valid, cfg.n_max, replace=False))
    N = len(sel)
    Yt = np.stack([np.log10(y15[sel, c]) if tr == "log10" else y15[sel, c] for _, c, tr, _ in PROPS], 1).astype(np.float32)
    print(f"selected {N} of {len(valid)} valid volumes (rule: {'all' if N == len(valid) else 'uniform random, seed %d' % cfg.data_seed})")

In [ ]:
# ============================== 4c. Reader workers and GPU featuriser ==============================
IO_SRC = '''
import numpy as np, h5py
_f = None; _path = None
def _init(path):
    global _path, _f
    _path = path; _f = None
def _vol(i):
    global _f
    if _f is None: _f = h5py.File(_path, "r")
    v = np.squeeze(np.asarray(_f["X"][int(i)]))
    hi = float(v.max())
    return v > (0.5 * hi if hi > 0 else 0.5)          # True where the stored value is "1"
def read_packed(i):
    b = _vol(i)
    return int(i), b.shape[0], np.packbits(b.ravel())
def connectivity(args):
    """6-connected pore clusters that span the sample in at least one axis: fraction of pore space they hold."""
    from scipy import ndimage as ndi
    i, ones_is_solid, k = args
    b = _vol(i); pore = ~b if ones_is_solid else b
    def stats(p):
        lab, n = ndi.label(p)
        if n == 0: return 0.0, False
        span = set()
        for ax in range(3):
            c = np.intersect1d(np.unique(np.take(lab, 0, axis=ax)), np.unique(np.take(lab, -1, axis=ax)))
            span.update(c[c > 0].tolist())
        if not span: return 0.0, False
        sizes = np.bincount(lab.ravel())
        return float(sizes[list(span)].sum() / max(int(p.sum()), 1)), True
    Dv = pore.shape[0]; out = [int(i), float(pore.mean())]
    out += list(stats(pore))
    for kk in (k, max(k // 2, 1)):
        fr = pore.reshape(Dv // kk, kk, Dv // kk, kk, Dv // kk, kk).mean((1, 3, 5))
        out += list(stats(fr >= 0.5)) + [float((fr >= 0.5).mean())]
    return out
'''
with open(os.path.join(LOCAL, "rock_io.py"), "w") as f: f.write(IO_SRC)
if LOCAL not in sys.path: sys.path.insert(0, LOCAL)
import importlib, rock_io; importlib.reload(rock_io)
import multiprocessing as mp
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)

@torch.no_grad()
def featurise(pore, ks, cap):
    """pore: [B,D,D,D] bool on DEVICE (True = pore), full resolution.
    Returns per-k coarse arrays (pore-fraction counts, pore-size map) and per-sample descriptors."""
    B, Dv = pore.shape[0], pore.shape[1]
    dt = torch.float16 if DEVICE == "cuda" else torch.float32
    m = pore[:, None].to(dt)
    phi = pore.float().mean((1, 2, 3))
    faces = ((pore[:, 1:] != pore[:, :-1]).sum((1, 2, 3)) + (pore[:, :, 1:] != pore[:, :, :-1]).sum((1, 2, 3))
             + (pore[:, :, :, 1:] != pore[:, :, :, :-1]).sum((1, 2, 3))).float()
    sv = faces / Dv**3                                              # pore-solid interface area per unit volume [1/px]
    dist, cur = m.clone(), m
    for it in range(cap):                                           # alternate 6- and 26-neighbour erosions (octagonal metric)
        if it % 2 == 0:
            cur = torch.minimum(torch.minimum(-F.max_pool3d(-cur, (3, 1, 1), 1, (1, 0, 0)),
                                              -F.max_pool3d(-cur, (1, 3, 1), 1, (0, 1, 0))),
                                -F.max_pool3d(-cur, (1, 1, 3), 1, (0, 0, 1)))
        else:
            cur = -F.max_pool3d(-cur, 3, 1, 1)
        dist += cur
    nb = cap + 2                                                    # distance values 0..cap+1 (0 = solid)
    off = torch.arange(B, device=pore.device).view(B, 1, 1, 1, 1) * nb
    hist = torch.bincount((dist.round().long() + off).flatten(), minlength=B * nb).view(B, nb).float()
    psd = hist[:, 1:] / hist[:, 1:].sum(1, keepdim=True).clamp(min=1)          # pore-size distribution
    r = torch.arange(1, nb, device=pore.device).float()
    cdf = psd.cumsum(1)
    r_mean = (psd * r).sum(1)
    r_p50 = r[(cdf >= 0.5).float().argmax(1)]; r_p90 = r[(cdf >= 0.9).float().argmax(1)]
    r_max = (psd > 0).float().mul(r).max(1).values
    het = F.avg_pool3d(pore[:, None].float(), Dv // 8).flatten(1).std(1)       # heterogeneity of porosity on an 8^3 grid
    desc = torch.cat([torch.stack([phi, sv, r_mean, r_p50, r_p90, r_max, het], 1), psd], 1)
    out = {}
    for k in ks:
        cnt = F.avg_pool3d(pore[:, None].float(), k) * (k**3)
        scale = k**3 if k**3 <= 255 else 255
        fr = (cnt if scale == k**3 else cnt / (k**3) * 255).round().to(torch.uint8)
        pm = F.max_pool3d(dist.float(), k).round().to(torch.uint8)
        out[k] = torch.cat([fr, pm], 1).cpu().numpy()               # [B,2,W,W,W] uint8
    return out, desc.cpu().numpy()
DESC_NAMES = ["phi", "sv", "r_mean", "r_p50", "r_p90", "r_max", "het"]

In [ ]:
# ============================== 4d. Build the working dataset ==============================
XW_PATH = os.path.join(D["proc"], "X_work.npy"); XH_PATH = os.path.join(D["proc"], "X_hi.npy")
if not os.path.exists(PROC_META):
    tic("preprocess")
    assert FULL_RES % cfg.work_res == 0 and FULL_RES % cfg.hi_res == 0
    K_W, K_H = FULL_RES // cfg.work_res, FULL_RES // cfg.hi_res
    ctx = mp.get_context("spawn")
    # --- phase convention: is the stored "1" solid or pore? Permeability must rise with pore fraction. ---
    probe = np.sort(np.random.default_rng(1).choice(N, min(48, N), replace=False))
    with ctx.Pool(min(N_WORKERS, 8), initializer=rock_io._init, initargs=(H5_PATH,)) as pool:
        fr1 = [np.unpackbits(pk)[:d**3].mean() for _, d, pk in pool.imap(rock_io.read_packed, sel[probe])]
    rho = st.spearmanr(fr1, Yt[probe, J_K])[0]
    ONES_IS_SOLID = bool(rho < 0) if abs(rho) > 0.2 else True      # DeePore README: 0 = void, 1 = solid
    print(f"phase check: Spearman(fraction of ones, log k) = {rho:+.2f} -> stored 1 = {'solid' if ONES_IS_SOLID else 'pore'}")
    # --- subset for the resolution study, drawn once ---
    rs = np.sort(np.random.default_rng(cfg.data_seed + 1).choice(N, min(cfg.res_subset_n, N), replace=False))
    res_pos = -np.ones(N, np.int64); res_pos[rs] = np.arange(len(rs))
    want_hi = cfg.run_resolution and cfg.run_hi_res
    XW = np.lib.format.open_memmap(XW_PATH, mode="w+", dtype=np.uint8, shape=(N, 2, cfg.work_res, cfg.work_res, cfg.work_res))
    XH = np.lib.format.open_memmap(XH_PATH, mode="w+", dtype=np.uint8, shape=(len(rs), 2, cfg.hi_res, cfg.hi_res, cfg.hi_res)) if want_hi else None
    DESC = np.zeros((N, len(DESC_NAMES) + cfg.poremap_cap + 1), np.float32)
    examples = {}
    BATCH = 8 if FULL_RES >= 200 else 64
    buf_i, buf_b = [], []
    def flush():
        if not buf_i: return
        ones = torch.from_numpy(np.stack(buf_b)).to(DEVICE)
        pore = ~ones if ONES_IS_SOLID else ones
        out, desc = featurise(pore, sorted({K_W, K_H}) if want_hi else [K_W], cfg.poremap_cap)
        ii = np.array(buf_i)
        XW[ii] = out[K_W]; DESC[ii] = desc
        if want_hi:
            mhi = res_pos[ii] >= 0
            if mhi.any(): XH[res_pos[ii][mhi]] = out[K_H][mhi]
        for b, i in enumerate(buf_i):
            if i < 6: examples[i] = pore[b, FULL_RES // 2].cpu().numpy()
        buf_i.clear(); buf_b.clear()
    with ctx.Pool(N_WORKERS, initializer=rock_io._init, initargs=(H5_PATH,)) as pool:
        for li, (gi, d, pk) in enumerate(tqdm(pool.imap(rock_io.read_packed, sel, chunksize=2), total=N, desc="featurise")):
            buf_i.append(li); buf_b.append(np.unpackbits(pk)[:d**3].reshape(d, d, d).astype(bool))
            if len(buf_i) == BATCH: flush()
        flush()
        # --- connectivity check on a subset (CPU) ---
        cs = np.sort(np.random.default_rng(cfg.data_seed + 2).choice(N, min(cfg.conn_subset_n, N), replace=False))
        conn = list(tqdm(pool.imap(rock_io.connectivity, [(int(sel[i]), ONES_IS_SOLID, K_W) for i in cs]), total=len(cs), desc="connectivity"))
    XW.flush(); del XW
    if want_hi: XH.flush(); del XH
    r1, r2 = FULL_RES // K_W, FULL_RES // max(K_W // 2, 1)
    conn_df = pd.DataFrame(conn, columns=["h5_index", "phi", "span_frac_full", "spans_full",
                                          f"span_frac_bin{r1}", f"spans_bin{r1}", f"phi_bin{r1}",
                                          f"span_frac_bin{r2}", f"spans_bin{r2}", f"phi_bin{r2}"])
    np.savez_compressed(os.path.join(D["proc"], "arrays.npz"), sel=sel, Yt=Yt, DESC=DESC, res_subset=rs,
                        ex_keys=np.array(sorted(examples)), ex_slices=np.stack([examples[k] for k in sorted(examples)]))
    conn_df.to_csv(os.path.join(D["proc"], "connectivity.csv"), index=False)
    excl_df.to_csv(os.path.join(D["proc"], "exclusions.csv"), index=False)
    t_pre = toc("preprocess")
    save_json(dict(N=N, N_ALL=N_ALL, FULL_RES=FULL_RES, K_W=K_W, K_H=K_H, ONES_IS_SOLID=ONES_IS_SOLID, phase_rho=float(rho),
                   source=DATA_SOURCE, md5=md5, has_hi=bool(want_hi), preprocess_s=t_pre, poremap_cap=cfg.poremap_cap,
                   work_res=cfg.work_res, hi_res=cfg.hi_res, n_max=cfg.n_max), PROC_META)
    if DRIVE_OK:                                                   # cache processed data on Drive if there is room
        need = sum(os.path.getsize(os.path.join(D["proc"], f)) for f in os.listdir(D["proc"]))
        if shutil.disk_usage(ROOT).free > 1.3 * need:
            print(f"caching processed data to Drive ({need/2**30:.1f} GB) ..."); shutil.copytree(D["proc"], D["proc_cache"], dirs_exist_ok=True)
        else:
            print("not enough free Drive space to cache processed data; it will be rebuilt after a disconnect.")

META = load_json(PROC_META)
assert META["work_res"] == cfg.work_res and META["n_max"] == cfg.n_max, "cached data was built with another configuration; delete " + D["proc"]
_a = np.load(os.path.join(D["proc"], "arrays.npz"))
sel, Yt, DESC, RES_SUBSET = _a["sel"], _a["Yt"], _a["DESC"], _a["res_subset"]
EX_SLICES = _a["ex_slices"]
N, FULL_RES, K_W = META["N"], META["FULL_RES"], META["K_W"]
conn_df = pd.read_csv(os.path.join(D["proc"], "connectivity.csv")); excl_df = pd.read_csv(os.path.join(D["proc"], "exclusions.csv"))
FRAC_SCALE = float(K_W**3 if K_W**3 <= 255 else 255); PORE_SCALE = float(cfg.poremap_cap + 1)
FRAC_SCALE_H = float(META["K_H"]**3 if META["K_H"]**3 <= 255 else 255)
print(f"working dataset: {N} volumes, input 2 x {cfg.work_res}^3 (coarse-graining factor {K_W}); source: {META['source']}")
save_table(excl_df, "T01_exclusions")
ddf = pd.DataFrame(DESC[:, :len(DESC_NAMES)], columns=DESC_NAMES)
tdf = pd.DataFrame(Yt, columns=PNAMES)
save_table(pd.concat([tdf.describe().T[["mean", "std", "min", "max"]], ddf.describe().T[["mean", "std", "min", "max"]]]).reset_index().rename(columns={"index": "variable"}),
           "T02_target_and_descriptor_ranges")

In [ ]:
# ============================== 4e. Put the working data where training can reach it fast ==============================
_xw = np.load(XW_PATH, mmap_mode="r")
ON_GPU = DEVICE == "cuda" and _xw.nbytes < 0.55 * VRAM_GB * 2**30
if ON_GPU:
    XW_T = torch.empty(_xw.shape, dtype=torch.uint8, device=DEVICE)
    for s in range(0, N, 1024): XW_T[s:s + 1024] = torch.from_numpy(np.ascontiguousarray(_xw[s:s + 1024])).to(DEVICE)
else:
    XW_T = torch.from_numpy(np.ascontiguousarray(_xw))
print(f"working array {_xw.nbytes/2**30:.2f} GB held on {'GPU' if ON_GPU else 'CPU RAM'}")
XH_MM, RES_POS = None, -np.ones(N, np.int64)                    # high-res subset stays on disk (memory-mapped)
if META["has_hi"] and cfg.run_resolution and cfg.run_hi_res:
    XH_MM = np.load(XH_PATH, mmap_mode="r"); RES_POS[RES_SUBSET] = np.arange(len(RES_SUBSET))
Y_T = torch.from_numpy(Yt).to(DEVICE)
PHI = DESC[:, 0].astype(np.float64)

In [ ]:
# ============================== Figure 1: data overview ==============================
def fig01():
    ne = min(3, len(EX_SLICES))
    fig = plt.figure(figsize=(11, 5.6)); gs = fig.add_gridspec(3, 6, width_ratios=[1, 1, 1, 1.25, 1.25, 1.25], wspace=0.35, hspace=0.45)
    xw = np.load(XW_PATH, mmap_mode="r")
    for r in range(ne):
        a0 = fig.add_subplot(gs[r, 0]); a0.imshow(EX_SLICES[r], cmap="gray_r", interpolation="nearest")
        a1 = fig.add_subplot(gs[r, 1]); a1.imshow(xw[r, 0, cfg.work_res // 2] / FRAC_SCALE, cmap=SEQ, vmin=0, vmax=1, interpolation="nearest")
        a2 = fig.add_subplot(gs[r, 2]); a2.imshow(xw[r, 1, cfg.work_res // 2] / PORE_SCALE, cmap=SEQ, vmin=0, interpolation="nearest")
        for a in (a0, a1, a2): a.set_xticks([]); a.set_yticks([]); a.grid(False)
        if r == 0:
            a0.set_title(f"binary, {FULL_RES}$^3$\n(pore = dark)"); a1.set_title(f"pore fraction, {cfg.work_res}$^3$"); a2.set_title(f"local pore size, {cfg.work_res}$^3$")
    for j, pn in enumerate(PNAMES):
        ax = fig.add_subplot(gs[0 if j < 3 else 2, 3 + j % 3])
        ax.hist(Yt[:, j], bins=50, color=MCOL["split"], edgecolor=SURF, linewidth=0.3)
        ax.set_xlabel(PAXIS[pn]); ax.set_yticks([]); ax.grid(False)
        ax.set_title(PSHORT[pn] if j else f"{PSHORT[pn]}   (n = {N:,} volumes)", fontweight="normal", fontsize=8.5)
    ax = fig.add_subplot(gs[1, 3:]); ax.axis("off")
    ax.text(0, 0.5, f"Input: two channels at {cfg.work_res}$^3$ computed from the full {FULL_RES}$^3$ volume (no crop, no re-binarisation).\n"
                    f"Labels: pore-network properties of the same {FULL_RES}$^3$ volume, lengths in voxels of that grid.",
            va="center", fontsize=8.5, color=INK2)
    savefig(fig, "F01_data_overview")
fig01()

## 5. Splits, morphology families and the run plan

**Provenance (Reviewer 2, comment 2).** DeePore volumes are generated by combining natural textures with augmentation, and the public file carries no
parent-rock identifier. A random split can therefore place close relatives in training and test. We deal with this in three ways and report all three:

- **Random splits** (main benchmark): exchangeable *within this collection*. This is the setting in which the conformal guarantee is stated.
- **Index-block split**: contiguous blocks of the file are kept together, a proxy for shared generation history. The block length is set from the
  measured autocorrelation of a label-free descriptor along the file index.
- **Morphology-family holdout**: volumes are clustered on label-free image descriptors (porosity, surface density, pore-size distribution, heterogeneity);
  whole families are withheld. This is a shift defined on the *inputs*, unlike the label-extrapolation test.

Splits: 55% training, 5% monitoring, 20% calibration, 20% test. The monitoring slice is only used to print a validation curve.
Training length is fixed in advance and no checkpoint is selected, so calibration and test data never influence the predictor.

In [ ]:
# ============================== 5a. Label-free descriptors and families ==============================
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor

_d = DESC.astype(np.float64); _eps = 1e-6
DFEAT_RAW = np.column_stack([np.log(_d[:, 0] / (1 - _d[:, 0] + _eps) + _eps), np.log(_d[:, 1] + _eps), np.log(_d[:, 2] + _eps),
                             np.log(_d[:, 4] + _eps), np.log(_d[:, 5] + _eps), _d[:, 6], np.sqrt(_d[:, len(DESC_NAMES):])])
DFEAT_RAW = DFEAT_RAW[:, DFEAT_RAW.std(0) > 1e-8]
DFEAT = PCA(n_components=min(8, DFEAT_RAW.shape[1]), random_state=0).fit_transform(StandardScaler().fit_transform(DFEAT_RAW))
DFEAT = DFEAT / DFEAT.std(0)                                     # label-free morphology coordinates used for families, leakage audit, OOD
FAM = KMeans(cfg.n_families, n_init=10, random_state=0).fit_predict(DFEAT)
_cent = np.array([DFEAT[FAM == f].mean(0) for f in range(cfg.n_families)])
_o1, _o2 = np.argsort(_cent[:, 0]), np.argsort(_cent[:, 1])
_nh = 2 if cfg.n_families >= 8 else 1
FAMILY_FOLDS = [list(_o1[:_nh]), list(_o1[-_nh:]), [f for f in list(_o2[-_nh - 1:]) if f not in list(_o1[:_nh]) + list(_o1[-_nh:])][:_nh]][:cfg.n_family_folds]
fam_df = pd.DataFrame([dict(family=f, n=int((FAM == f).sum()), **{k: float(DESC[FAM == f, i].mean()) for i, k in enumerate(DESC_NAMES)},
                            mean_log_k=float(Yt[FAM == f, J_K].mean()),
                            held_out_in_fold=",".join(str(i) for i, h in enumerate(FAMILY_FOLDS) if f in h) or "-") for f in range(cfg.n_families)])
save_table(fam_df, "T03_morphology_families")

# ---- structure along the file index (proxy for shared generation history) ----
_x = DFEAT[:, 0] - DFEAT[:, 0].mean(); LAGS = np.unique(np.round(np.geomspace(1, min(2000, N // 4), 40)).astype(int))
ACF = np.array([float((_x[:-l] * _x[l:]).mean() / _x.var()) for l in LAGS])
_below = np.where(ACF < 0.2)[0]
BLOCK = int(np.clip(2 * (LAGS[_below[0]] if len(_below) else LAGS[-1]), 20, 500)) if ACF[0] >= 0.2 else 20
print(f"index autocorrelation of morphology PC1: lag-1 = {ACF[0]:.2f}; block length used for the grouped split: {BLOCK}")

In [ ]:
# ============================== 5b. Run plan ==============================
FR = (cfg.frac_train, cfg.frac_val, cfg.frac_cal, cfg.frac_test)
def split_pool(pool, seed, groups=None):
    """Random split of `pool` into tr/val/cal/te. If `groups` is given, whole groups are kept together."""
    rng = np.random.default_rng(1000 + seed)
    if groups is None:
        p = rng.permutation(pool); n = len(p)
        a = int(round(FR[0] * n)); b = a + int(round(FR[1] * n)); c = b + int(round(FR[2] * n))
        parts = [p[:a], p[a:b], p[b:c], p[c:]]
    else:
        ug = rng.permutation(np.unique(groups[pool])); cut = np.cumsum(FR) * len(ug)
        gid = np.searchsorted(cut, np.arange(len(ug)) + 0.5); lut = dict(zip(ug.tolist(), gid.tolist()))
        lab = np.array([lut[g] for g in groups[pool].tolist()]); parts = [pool[lab == i] for i in range(4)]
    return dict(tr=np.sort(parts[0]), val=np.sort(parts[1]), cal=np.sort(parts[2]), te=np.sort(parts[3]))

RUNS = {}
def reg(tag, split, mode="joint", variant="main", init_seed=0, group="main", note=""):
    RUNS[tag] = dict(tag=tag, split=split, mode=mode, variant=variant, init_seed=init_seed, group=group, note=note)
ALL = np.arange(N)
SPLITS = {s: split_pool(ALL, s) for s in range(cfg.n_seeds)}
for s in range(cfg.n_seeds): reg(f"main_s{s}", SPLITS[s], "joint", "main", s, "main")
if cfg.run_ablations:
    for s in range(min(cfg.n_abl, cfg.n_seeds)):
        reg(f"meanonly_s{s}", SPLITS[s], "mean_only", "main", s, "ablation", "same split and initial seed as main_s%d" % s)
        reg(f"gauss_s{s}_m0", SPLITS[s], "gaussian", "main", s, "ablation")
    for m in range(1, cfg.ens_M): reg(f"gauss_s0_m{m}", SPLITS[0], "gaussian", "main", 1000 + m, "ablation", "deep-ensemble member")
if cfg.run_shift:
    thrA = float(np.quantile(Yt[:, J_K], 1 - cfg.ood_top_frac)); oodA = Yt[:, J_K] >= thrA
    sp = split_pool(ALL[~oodA], 0); sp["ood"] = ALL[oodA]; reg("shiftA", sp, group="shift", note=f"label-support extrapolation: log k >= {thrA:.3f} withheld")
    for i, held in enumerate(FAMILY_FOLDS):
        o = np.isin(FAM, held); sp = split_pool(ALL[~o], 0); sp["ood"] = ALL[o]
        reg(f"family_f{i}", sp, group="shift", note=f"morphology families {held} withheld")
    reg("block", split_pool(ALL, 0, groups=ALL // BLOCK), group="shift", note=f"index blocks of {BLOCK} kept together")
if cfg.run_resolution:
    spr = split_pool(RES_SUBSET, 0)
    for v in ["main", "frac", "bin", "lo"] + (["hi"] if (cfg.run_hi_res and XH_MM is not None) else []):
        reg(f"res_{v}", spr, "mean_only", v, 0, "resolution")
for tag, r in RUNS.items(): np.savez_compressed(os.path.join(D["splits"], tag + ".npz"), **r["split"], h5_index=sel)
plan = pd.DataFrame([dict(run=t, group=r["group"], mode=r["mode"], input=r["variant"], init_seed=r["init_seed"],
                          **{k: len(v) for k, v in r["split"].items()}, note=r["note"]) for t, r in RUNS.items()]).fillna(0)
save_table(plan, "T04_run_plan_and_sample_counts")
print(len(RUNS), "training runs planned")

In [ ]:
# ============================== Figure 2: provenance structure and shift folds ==============================
def fig02():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.7), gridspec_kw=dict(width_ratios=[1.15, 1, 1]))
    ax = axes[0]; held_any = np.zeros(N, bool)
    ax.scatter(DFEAT[:, 0], DFEAT[:, 1], s=2, color="#c9c8c2", lw=0, rasterized=True, label="kept in training")
    fold_cols = ["#2a78d6", "#eb6834", "#1baf7a"]
    for i, held in enumerate(FAMILY_FOLDS):
        m = np.isin(FAM, held); ax.scatter(DFEAT[m, 0], DFEAT[m, 1], s=2.5, color=fold_cols[i], lw=0, rasterized=True, label=f"withheld in fold {i}")
    ax.set_xlabel("morphology PC1 (label-free)"); ax.set_ylabel("morphology PC2"); ax.set_title("Morphology families withheld")
    ax.legend(markerscale=4, loc="best", fontsize=7.5)
    ax = axes[1]; ax.plot(LAGS, ACF, color=MCOL["split"], marker="o", ms=3); ax.axhline(0.2, color=INK2, lw=0.8, ls=(0, (4, 3)))
    ax.axvline(BLOCK / 2, color=INK2, lw=0.8); ax.set_xscale("log"); ax.set_xlabel("lag along file index"); ax.set_ylabel("autocorrelation of PC1")
    ax.set_title("Structure along the file index"); ax.text(BLOCK / 2 * 1.1, 0.9 * max(ACF.max(), 0.3), f"block = {BLOCK}", fontsize=7.5, color=INK2)
    ax = axes[2]; bins = np.linspace(Yt[:, J_K].min(), Yt[:, J_K].max(), 40)
    ax.hist(Yt[:, J_K], bins=bins, color="#c9c8c2", label="all volumes")
    for i, held in enumerate(FAMILY_FOLDS):
        ax.hist(Yt[np.isin(FAM, held), J_K], bins=bins, histtype="step", lw=1.6, color=fold_cols[i], label=f"fold {i}")
    ax.axvline(np.quantile(Yt[:, J_K], 1 - cfg.ood_top_frac), color=INK, lw=1, ls=(0, (4, 3)))
    ax.text(np.quantile(Yt[:, J_K], 1 - cfg.ood_top_frac), ax.get_ylim()[1] * 0.95, " label cut-off", fontsize=7.5, va="top")
    ax.set_xlabel(PLABEL["log_k"]); ax.set_ylabel("count"); ax.set_title("Permeability of withheld sets"); ax.legend(fontsize=7.5)
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F02_provenance_and_shift_folds")
fig02()

## 6. Network, training and prediction

- **Backbone**: 3D residual CNN, three stride-2 stages, global average pooling, one fully connected layer.
- **Heads**: mean; log-variance (heteroscedastic Gaussian); ten quantiles (0.01 to 0.99) so that CQR has a matching pair at each target level.
- **Training modes**
  - `joint`: MSE + 0.5·Gaussian NLL + pinball loss, all through the shared backbone (the design of the original submission).
    The auxiliary losses **do** change the shared representation; the `mean_only` ablation measures by how much (Reviewer 2, comment 9).
  - `mean_only`: MSE only.
  - `gaussian`: Gaussian NLL only, end to end (dedicated baseline; five members form the deep ensemble).
- **Inference modes**: batch normalisation always uses running statistics. Dropout is **off** for every deterministic output
  (point prediction, Gaussian head, quantile heads) and **on** only during the MC-dropout passes.
- **Augmentation**: random mirror flips only. Axis permutations are not used because directional labels are not guaranteed to be isotropic.

In [ ]:
# ============================== 6a. Model ==============================
class ResBlock(nn.Module):
    def __init__(self, c, p):
        super().__init__()
        self.c1 = nn.Conv3d(c, c, 3, padding=1, bias=False); self.b1 = nn.BatchNorm3d(c)
        self.c2 = nn.Conv3d(c, c, 3, padding=1, bias=False); self.b2 = nn.BatchNorm3d(c)
        self.drop = nn.Dropout3d(p)
    def forward(self, x):
        h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(self.drop(h)))
        return F.silu(x + h)
def down(ci, co): return nn.Sequential(nn.Conv3d(ci, co, 3, stride=2, padding=1, bias=False), nn.BatchNorm3d(co), nn.SiLU())
class RockNet(nn.Module):
    def __init__(self, in_ch, n_out, n_q, base=16, p=0.1):
        super().__init__()
        b = base
        self.stem = nn.Sequential(nn.Conv3d(in_ch, b, 3, padding=1, bias=False), nn.BatchNorm3d(b), nn.SiLU())
        self.s1 = nn.Sequential(down(b, 2 * b), ResBlock(2 * b, p))
        self.s2 = nn.Sequential(down(2 * b, 4 * b), ResBlock(4 * b, p))
        self.s3 = nn.Sequential(down(4 * b, 8 * b), ResBlock(8 * b, p))
        self.fc = nn.Sequential(nn.Linear(8 * b, 8 * b), nn.SiLU()); self.drop = nn.Dropout(p)
        self.h_mu = nn.Linear(8 * b, n_out); self.h_lv = nn.Linear(8 * b, n_out); self.h_q = nn.Linear(8 * b, n_out * n_q)
        self.n_out, self.n_q = n_out, n_q
    def forward(self, x):
        z = self.fc(self.s3(self.s2(self.s1(self.stem(x)))).mean((2, 3, 4)))
        zd = self.drop(z)
        return self.h_mu(zd), self.h_lv(zd).clamp(-10, 6), self.h_q(zd).view(-1, self.n_out, self.n_q), z
def set_mc_dropout(model, on):
    model.eval()
    if on:
        for m in model.modules():
            if isinstance(m, (nn.Dropout, nn.Dropout3d)): m.train()
IN_CH = {"main": 2, "lo": 2, "hi": 2, "frac": 1, "bin": 1}
N_PARAMS = sum(p.numel() for p in RockNet(2, P, N_Q, cfg.base, cfg.p_drop).parameters())
print(f"RockNet: {N_PARAMS/1e6:.3f} M trainable parameters (base width {cfg.base})")

def make_batch(idx, variant="main", train=False):
    """idx: 1-D LongTensor of dataset rows. Returns a float input batch on DEVICE."""
    if variant == "hi":
        pos = RES_POS[idx.cpu().numpy()]; o = np.argsort(pos)
        xb = np.empty((len(pos),) + XH_MM.shape[1:], np.uint8); xb[o] = XH_MM[pos[o]]
        x = torch.from_numpy(xb).to(DEVICE).float()
    else:
        x = XW_T[idx.to(XW_T.device)].to(DEVICE, non_blocking=True).float()
    x0, x1 = x[:, 0:1] / (FRAC_SCALE_H if variant == "hi" else FRAC_SCALE), x[:, 1:2] / PORE_SCALE
    if variant in ("main", "hi"): x = torch.cat([x0, x1], 1)
    elif variant == "lo": x = torch.cat([F.avg_pool3d(x0, 2), F.max_pool3d(x1, 2)], 1)
    elif variant == "frac": x = x0
    elif variant == "bin": x = (x0 >= 0.5).float()                  # re-binarised input, as in the original submission
    if train:
        flips = [d for d in (2, 3, 4) if torch.rand(()) < 0.5]
        if flips: x = torch.flip(x, flips)
    return x.contiguous(memory_format=torch.channels_last_3d)

TAUS = torch.tensor(Q_LEVELS, device=DEVICE).view(1, 1, -1)
def loss_fn(out, y, mode, warm=False):
    mu, lv, q, _ = out; mu, lv, q = mu.float(), lv.float(), q.float()
    mse = F.mse_loss(mu, y)
    if mode == "mean_only": return mse
    nll = 0.5 * (torch.exp(-lv) * (y - mu)**2 + lv).mean()
    if mode == "gaussian": return nll + (mse if warm else 0.0)       # short MSE warm-up stabilises the pure-NLL start
    e = y.unsqueeze(-1) - q
    return mse + cfg.w_nll * nll + cfg.w_q * torch.maximum(TAUS * e, (TAUS - 1) * e).mean()

In [ ]:
# ============================== 6b. Training (resumable) and prediction (cached) ==============================
import contextlib
AMP = DEVICE == "cuda"
def ac(): return torch.autocast("cuda", dtype=torch.float16) if AMP else contextlib.nullcontext()
def train_run(tag):
    r = RUNS[tag]; ck = os.path.join(D["ckpt"], tag + ".pt"); sp = r["split"]
    tr = torch.from_numpy(sp["tr"]); va = torch.from_numpy(sp["val"])
    ym, ys = Y_T[tr.to(DEVICE)].mean(0), Y_T[tr.to(DEVICE)].std(0) + 1e-8
    set_seed(10_000 + r["init_seed"])
    model = RockNet(IN_CH[r["variant"]], P, N_Q, cfg.base, cfg.p_drop).to(DEVICE).to(memory_format=torch.channels_last_3d)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    bs = max(8, cfg.batch // 4) if r["variant"] == "hi" else cfg.batch
    steps = max(1, len(tr) // bs); total = steps * cfg.epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=total, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    state = dict(epoch=0, hist=[], train_s=0.0)
    if os.path.exists(ck):
        c = torch.load(ck, map_location=DEVICE, weights_only=False)
        model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"]); scaler.load_state_dict(c["scaler"])
        state = c["state"]
    g = torch.Generator().manual_seed(20_000 + r["init_seed"] + 97 * state["epoch"])
    for ep in range(state["epoch"], cfg.epochs):
        t0 = time.time(); model.train(); perm = tr[torch.randperm(len(tr), generator=g)]; tl = 0.0
        for b in range(steps):
            idx = perm[b * bs:(b + 1) * bs]
            x = make_batch(idx, r["variant"], train=True); y = (Y_T[idx.to(DEVICE)] - ym) / ys
            with ac():
                out = model(x)
            loss = loss_fn(out, y, r["mode"], warm=(ep * steps + b) < 0.1 * total)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()
        model.eval(); se = 0.0
        with torch.no_grad():
            vb = 16 if r["variant"] == "hi" else 128
            for b in range(0, len(va), vb):
                idx = va[b:b + vb]
                with ac():
                    mu = model(make_batch(idx, r["variant"]))[0].float()
                se += ((mu - (Y_T[idx.to(DEVICE)] - ym) / ys)**2).sum().item()
        state["epoch"] = ep + 1; state["train_s"] += time.time() - t0
        state["hist"].append(dict(epoch=ep + 1, train_loss=tl / steps, val_mse_z=se / max(1, len(va) * P)))
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                        state=state, ym=ym.cpu(), ys=ys.cpu(), run={k: v for k, v in r.items() if k != "split"}), ck)
    return model, ym, ys, state

@torch.no_grad()
def predict_run(tag, model, ym, ys, mc_grid=None):
    """Deterministic outputs for every split part; MC-dropout for val/cal/test/ood. Everything is returned in target units."""
    r = RUNS[tag]; out = {}; t_det = t_mc = 0.0; n_det = n_mc = 0
    ymn, ysn = ym.cpu().numpy(), ys.cpu().numpy()
    for part, ids in r["split"].items():
        if len(ids) == 0: continue
        it = torch.from_numpy(ids); bsz = 16 if r["variant"] == "hi" else 256
        acc = {k: [] for k in ["mu", "sig", "q", "feat"]}
        set_mc_dropout(model, False); t0 = time.time()
        for b in range(0, len(it), bsz):
            with ac():
                mu, lv, q, z = model(make_batch(it[b:b + bsz], r["variant"]))
            acc["mu"].append(mu.float().cpu()); acc["sig"].append(torch.exp(0.5 * lv.float()).cpu()); acc["q"].append(q.float().cpu()); acc["feat"].append(z.float().cpu())
        if DEVICE == "cuda": torch.cuda.synchronize()
        t_det += time.time() - t0; n_det += len(it)
        o = {k: torch.cat(v).numpy() for k, v in acc.items()}
        res = dict(idx=ids, y=Yt[ids], mu=o["mu"] * ysn + ymn, sig=o["sig"] * ysn, q=o["q"] * ysn[None, :, None] + ymn[None, :, None],
                   feat=o["feat"].astype(np.float16))
        if part != "tr" and r["mode"] == "joint" and r["group"] != "resolution":
            Tmax = max(max(mc_grid), cfg.mc_T) if mc_grid else cfg.mc_T
            s1 = np.zeros((len(ids), P)); s2 = np.zeros((len(ids), P)); sv = np.zeros((len(ids), P))
            set_mc_dropout(model, True); t0 = time.time()
            for t in range(1, Tmax + 1):
                mus, vars_ = [], []
                for b in range(0, len(it), bsz):
                    with ac():
                        mu, lv, _, _ = model(make_batch(it[b:b + bsz], r["variant"]))
                    mus.append(mu.float().cpu()); vars_.append(torch.exp(lv.float()).cpu())
                m_ = torch.cat(mus).numpy() * ysn + ymn; s1 += m_; s2 += m_**2; sv += torch.cat(vars_).numpy() * ysn**2
                if (mc_grid and t in mc_grid) or t == cfg.mc_T:
                    snap = dict(mc_mu=s1 / t, mc_vm=np.maximum(s2 / t - (s1 / t)**2, 0) * (t / max(t - 1, 1)), mc_mv=sv / t)
                    if t == cfg.mc_T: res.update(snap)
                    if mc_grid and t in mc_grid: res.update({f"{k}_T{t}": v for k, v in snap.items()})
            if DEVICE == "cuda": torch.cuda.synchronize()
            t_mc += time.time() - t0; n_mc += len(it) * Tmax
            set_mc_dropout(model, False)
        out[part] = res
    timing = dict(ms_per_volume_deterministic=1e3 * t_det / max(n_det, 1), ms_per_volume_per_mc_pass=1e3 * t_mc / max(n_mc, 1))
    return out, timing

def save_preds(tag, out, meta):
    flat = {f"{part}__{k}": v for part, d in out.items() for k, v in d.items()}
    np.savez_compressed(os.path.join(D["preds"], tag + ".npz"), **flat); save_json(meta, os.path.join(D["logs"], tag + ".json"))
_PRED_CACHE = {}
def load_preds(tag):
    if tag not in _PRED_CACHE:
        z = np.load(os.path.join(D["preds"], tag + ".npz")); out = {}
        for k in z.files:
            part, name = k.split("__", 1); out.setdefault(part, {})[name] = z[k].astype(np.float64) if z[k].dtype.kind == "f" else z[k]
        _PRED_CACHE[tag] = out
    return _PRED_CACHE[tag]
def run_log(tag): return load_json(os.path.join(D["logs"], tag + ".json"))

In [ ]:
# ============================== 6c. Execute the run plan ==============================
order = sorted(RUNS, key=lambda t: ["main", "ablation", "shift", "resolution"].index(RUNS[t]["group"]))
todo = [t for t in order if not (os.path.exists(os.path.join(D["preds"], t + ".npz")) and os.path.exists(os.path.join(D["logs"], t + ".json")))]
print(f"{len(order) - len(todo)} runs already cached, {len(todo)} to do")
t_first = None
for i, tag in enumerate(todo):
    model, ym, ys, state = train_run(tag)
    out, timing = predict_run(tag, model, ym, ys, mc_grid=cfg.mc_T_grid if tag == "main_s0" else None)
    r = RUNS[tag]
    save_preds(tag, out, dict(tag=tag, mode=r["mode"], variant=r["variant"], group=r["group"], init_seed=r["init_seed"], note=r["note"],
                              n={k: len(v) for k, v in r["split"].items()}, train_minutes=state["train_s"] / 60, hist=state["hist"],
                              ym=ym.cpu().numpy(), ys=ys.cpu().numpy(), **timing, device=GPU_NAME, n_params=N_PARAMS))
    del model
    if DEVICE == "cuda": torch.cuda.empty_cache()
    h = state["hist"][-1]
    msg = f"[{i+1}/{len(todo)}] {tag}: {state['train_s']/60:.1f} min train, val MSE(z) {h['val_mse_z']:.4f}"
    if i == 0 and len(todo) > 1:
        t_first = state["train_s"] * 1.25
        msg += f" | rough projection for the remaining {len(todo)-1} runs: {t_first*(len(todo)-1)/3600:.1f} h (full-size runs; subset runs are faster)"
    print(msg)
print("all runs available")

## 7. Conformal predictors: definitions used in this notebook

All intervals are built **per property** (marginal, property-wise coverage). Joint coverage over several properties is a separate construction in Section 12.

**Finite-sample quantile.** For calibration scores $s_1,\dots,s_n$ and miscoverage $\alpha$, let $k=\lceil (n+1)(1-\alpha)\rceil$.
The threshold is the $k$-th smallest score, $\hat q = s_{(k)}$, and $\hat q=+\infty$ if $k>n$. Ties are kept as they are (the guarantee still holds and is conservative).

| Method | Score on calibration | Interval for a new $x$ | Needs from the predictor |
|---|---|---|---|
| Split | $\lvert y-\hat\mu(x)\rvert$ | $[\hat\mu(x)-\hat q,\ \hat\mu(x)+\hat q]$ | a point prediction only |
| Normalized | $\lvert y-\hat\mu(x)\rvert/(\hat\sigma(x)+\varepsilon)$ | $[\hat\mu(x)-\hat q(\hat\sigma(x)+\varepsilon),\ \hat\mu(x)+\hat q(\hat\sigma(x)+\varepsilon)]$ | plus a scale estimate $\hat\sigma$ |
| CQR | $\max\{\hat q_{lo}(x)-y,\ y-\hat q_{hi}(x)\}$ | $[\hat q_{lo}(x)-\hat q,\ \hat q_{hi}(x)+\hat q]$ | plus two quantile predictions |
| Mondrian | $\lvert y-\hat\mu(x)\rvert$ within group $g(x)$ | $[\hat\mu(x)-\hat q_{g(x)},\ \hat\mu(x)+\hat q_{g(x)}]$ | a point prediction and a fixed grouping |

Details that matter for validity and reproducibility:

- **Only split conformal is strictly post hoc.** Normalized needs a scale estimate and CQR needs quantile outputs; if a model lacks them they must be fitted.
- **Mondrian groups** are bins of the predicted value of the *same* property. The bin edges are quantiles of the **training-set predictions**, so the partition is fixed
  before calibration data are seen and the group-conditional guarantee $P(y\in C(x)\mid g(x)=g)\ge 1-\alpha$ holds exactly. A group with too few calibration points for a finite
  threshold falls back to the pooled threshold, and the number of such cases is reported.
- **CQR levels.** The network has quantile heads at 0.01, 0.025, 0.05, 0.10, 0.15 and their complements. For a target level the pair with the closest nominal coverage is used.
  Crossed quantile heads are sorted. A negative $\hat q$ shrinks the band; if it inverts the band the interval collapses to its midpoint (zero width, counted as a miss unless $y$ equals it), and the count is reported.
- **$\varepsilon$** is `eps_rel` times the training standard deviation of the target.
- Empirical coverage close to the target is *consistent with* the guarantee. It does not prove it. Section 10 therefore compares the **distribution** of coverage over
  random calibration/test partitions with the exact beta-binomial law that a correct implementation must follow.

In [ ]:
# ============================== 7a. Conformal library ==============================
def cq(scores, alpha):
    """k-th smallest calibration score with k = ceil((n+1)(1-alpha)); +inf if k > n."""
    s = np.asarray(scores, float); n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha) - 1e-9))
    return np.inf if (n == 0 or k > n) else float(np.partition(s, k - 1)[k - 1])
def zq(alpha): return float(st.norm.ppf(1 - alpha / 2))
def q_pair(alpha):
    j = int(np.argmin([abs((1 - 2 * t) - (1 - alpha)) for t in cfg.q_lo])); return j, j + len(cfg.q_lo)
def group_edges(tr_mu, G): return np.quantile(tr_mu, np.linspace(0, 1, G + 1)[1:-1])

def interval(method, cal, te, j, alpha, tr=None, eps_rel=None, G=None):
    """Returns lo, hi, info for property j. cal / te / tr are prediction dicts in target units."""
    eps_rel = cfg.eps_rel if eps_rel is None else eps_rel; G = cfg.n_groups if G is None else G
    yc, mc_, mt = cal["y"][:, j], cal["mu"][:, j], te["mu"][:, j]; info = {}
    if method == "split":
        q = cq(np.abs(yc - mc_), alpha); lo, hi = mt - q, mt + q
    elif method == "norm":
        eps = eps_rel * tr["y"][:, j].std()
        q = cq(np.abs(yc - mc_) / (cal["sig"][:, j] + eps), alpha); w = q * (te["sig"][:, j] + eps); lo, hi = mt - w, mt + w
    elif method == "cqr":
        a, b = q_pair(alpha)
        cl, ch = np.minimum(cal["q"][:, j, a], cal["q"][:, j, b]), np.maximum(cal["q"][:, j, a], cal["q"][:, j, b])
        tl, th = np.minimum(te["q"][:, j, a], te["q"][:, j, b]), np.maximum(te["q"][:, j, a], te["q"][:, j, b])
        q = cq(np.maximum(cl - yc, yc - ch), alpha); lo, hi = tl - q, th + q
        inv = lo > hi; mid = 0.5 * (lo + hi); lo, hi = np.where(inv, mid, lo), np.where(inv, mid, hi)
        info = dict(n_inverted=int(inv.sum()), n_crossed=int((te["q"][:, j, a] > te["q"][:, j, b]).sum()), qhat=q)
    elif method == "cqr_fixed":                                   # original submission: 0.05/0.95 heads at every target level
        a, b = q_pair(0.10)
        cl, ch = np.minimum(cal["q"][:, j, a], cal["q"][:, j, b]), np.maximum(cal["q"][:, j, a], cal["q"][:, j, b])
        tl, th = np.minimum(te["q"][:, j, a], te["q"][:, j, b]), np.maximum(te["q"][:, j, a], te["q"][:, j, b])
        q = cq(np.maximum(cl - yc, yc - ch), alpha); lo, hi = tl - q, th + q
        inv = lo > hi; mid = 0.5 * (lo + hi); lo, hi = np.where(inv, mid, lo), np.where(inv, mid, hi)
    elif method == "mondrian":
        e = group_edges(tr["mu"][:, j], G); gc, gt = np.searchsorted(e, mc_, side="right"), np.searchsorted(e, mt, side="right")
        res = np.abs(yc - mc_); qg = np.empty(G); fb = 0; pooled = cq(res, alpha)
        for g in range(G):
            qg[g] = cq(res[gc == g], alpha)
            if not np.isfinite(qg[g]): qg[g] = pooled; fb += 1
        lo, hi = mt - qg[gt], mt + qg[gt]; info = dict(n_fallback_groups=fb, min_group_n=int(np.bincount(gc, minlength=G).min()))
    elif method == "gauss":
        w = zq(alpha) * te["sig"][:, j]; lo, hi = mt - w, mt + w
    elif method == "mcd":
        w = zq(alpha) * np.sqrt(te["mc_vm"][:, j]); lo, hi = te["mc_mu"][:, j] - w, te["mc_mu"][:, j] + w
    elif method == "mcd_tot":
        w = zq(alpha) * np.sqrt(te["mc_vm"][:, j] + te["mc_mv"][:, j]); lo, hi = te["mc_mu"][:, j] - w, te["mc_mu"][:, j] + w
    else:
        raise ValueError(method)
    return lo, hi, info
def cov(y, lo, hi): return float(np.mean((y >= lo) & (y <= hi)))
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z**2 / n; c = (p + z**2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h
def cov_law(n_cal, n_test, alpha):
    """Exact mean and sd of empirical test coverage of split conformal with continuous scores (beta-binomial law)."""
    k = int(np.ceil((n_cal + 1) * (1 - alpha) - 1e-9)); a, b = k, n_cal + 1 - k
    m = a / (a + b); v = a * b / ((a + b)**2 * (a + b + 1))
    return m, float(np.sqrt(v + (m - (v + m**2)) / n_test))
def ensemble_pred(tags, part):
    mus = np.stack([load_preds(t)[part]["mu"] for t in tags]); sig = np.stack([load_preds(t)[part]["sig"] for t in tags])
    mu = mus.mean(0); return dict(mu=mu, sig=np.sqrt(np.maximum((sig**2 + mus**2).mean(0) - mu**2, 0)), y=load_preds(tags[0])[part]["y"])

# ---- self-test of the conformal routines on a case with a known answer ----
_r = np.random.default_rng(0); _c = []
for _ in range(400):
    e = _r.standard_normal(1200); q = cq(np.abs(e[:400]), 0.1); _c.append(np.mean(np.abs(e[400:]) <= q))
_m, _s = cov_law(400, 800, 0.1)
assert abs(np.mean(_c) - _m) < 4 * _s / np.sqrt(400) and abs(np.std(_c) / _s - 1) < 0.15, "conformal self-test failed"
print(f"self-test passed: simulated coverage {np.mean(_c):.4f} +/- {np.std(_c):.4f}; exact law {_m:.4f} +/- {_s:.4f}")
def reg_metrics(y, p):
    e = p - y; return dict(MAE=float(np.abs(e).mean()), RMSE=float(np.sqrt((e**2).mean())), R2=float(1 - (e**2).sum() / ((y - y.mean())**2).sum()))
MAIN = [f"main_s{s}" for s in range(cfg.n_seeds)]

## 8. Does the input still describe the labelled geometry? (Reviewer 2, comments 1 and 11)

Three checks, all on the same held-out subset and split:

1. **Connectivity.** For a random subset of full-resolution volumes we label 6-connected pore clusters and record whether the pore space spans the sample.
   We then re-binarise the same volumes at the working resolution by majority vote (what a plain down-sample does) and repeat. If spanning changes,
   a re-binarised input is no longer the geometry the permeability label was computed on.
2. **Porosity is not a learning problem.** The pore-fraction channel conserves porosity exactly, so any network "porosity prediction" is redundant.
   Porosity is therefore not a target here; it is used only as a label-free grouping and shift variable.
3. **Input ablation.** The same network trained on (i) our two-channel input, (ii) pore fraction only, (iii) a re-binarised input as in the original submission,
   (iv) half resolution and (v) double resolution.

In [ ]:
# ============================== 8a. Connectivity and porosity under re-binarisation ==============================
wr, hr = FULL_RES // K_W, FULL_RES // max(K_W // 2, 1)        # resolutions at which connectivity was re-checked
c = conn_df
conn_rows = [dict(check=f"pore space spans the sample at {FULL_RES}^3", value=c.spans_full.mean()),
             dict(check=f"... and still spans after majority re-binarisation to {hr}^3", value=(c.spans_full & c[f"spans_bin{hr}"]).sum() / max(c.spans_full.sum(), 1)),
             dict(check=f"... and still spans after majority re-binarisation to {wr}^3", value=(c.spans_full & c[f"spans_bin{wr}"]).sum() / max(c.spans_full.sum(), 1)),
             dict(check=f"spanning status changed by re-binarisation to {wr}^3 (any direction)", value=(c.spans_full != c[f"spans_bin{wr}"]).mean()),
             dict(check=f"median |porosity change| from re-binarisation to {wr}^3", value=(c[f"phi_bin{wr}"] - c.phi).abs().median()),
             dict(check=f"max |porosity change| from re-binarisation to {wr}^3", value=(c[f"phi_bin{wr}"] - c.phi).abs().max()),
             dict(check="max |porosity change| in the two-channel input used here", value=POR_ERR if "POR_ERR" in dir() else np.nan)]
_xw = np.load(XW_PATH, mmap_mode="r"); _ii = np.arange(0, N, max(1, N // 400))
POR_ERR = float(np.abs(np.array([_xw[i, 0].mean() / FRAC_SCALE for i in _ii]) - DESC[_ii, 0]).max()); conn_rows[-1]["value"] = POR_ERR
save_table(pd.DataFrame(conn_rows), "T12_connectivity_and_porosity_checks")

In [ ]:
# ============================== 8b. Input ablation ==============================
RES_V = [v for v in ["main", "frac", "bin", "lo", "hi"] if f"res_{v}" in RUNS]
RES_LAB = {"main": f"two-channel, {wr}$^3$ (used)", "frac": f"pore fraction only, {wr}$^3$", "bin": f"re-binarised, {wr}$^3$",
           "lo": f"two-channel, {cfg.work_res//2}$^3$", "hi": f"two-channel, {cfg.hi_res}$^3$"}
rows = []
for v in RES_V:
    Pr = load_preds(f"res_{v}"); lg = run_log(f"res_{v}")
    for j, pn in enumerate(PNAMES):
        q = cq(np.abs(Pr["cal"]["y"][:, j] - Pr["cal"]["mu"][:, j]), cfg.alpha)
        rows.append(dict(input=v, label=RES_LAB[v].replace("$^3$", "^3"), prop=pn, **reg_metrics(Pr["te"]["y"][:, j], Pr["te"]["mu"][:, j]),
                         split_width=2 * q, split_coverage=cov(Pr["te"]["y"][:, j], Pr["te"]["mu"][:, j] - q, Pr["te"]["mu"][:, j] + q),
                         train_minutes=lg["train_minutes"]))
RES = pd.DataFrame(rows); save_table(RES, "T13_input_representation_ablation")

def fig03():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.7), gridspec_kw=dict(width_ratios=[1.35, 1, 1]))
    ax = axes[0]; base = RES[RES.input == "main"].set_index("prop").RMSE
    vcol = {"frac": "#2a78d6", "bin": "#eb6834", "lo": "#1baf7a", "hi": "#6250d6"}; vmk = {"frac": "s", "bin": "X", "lo": "v", "hi": "^"}
    ax.axhline(1, color=INK, lw=0.9); ax.text(P - 0.5, 1, "used input", va="bottom", ha="right", fontsize=7.5, color=INK2)
    for i, v in enumerate([v for v in RES_V if v != "main"]):
        d = RES[RES.input == v].set_index("prop").RMSE.reindex(PNAMES) / base.reindex(PNAMES)
        ax.plot(np.arange(P) + (i - 1.5) * 0.12, d.values, ls="none", marker=vmk[v], ms=7, color=vcol[v], label=RES_LAB[v])
    prop_ticks(ax)
    ax.set_ylabel("test RMSE relative to the used input"); ax.set_title("Input ablation (same split)")
    ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    ax = axes[1]
    for flag, col, lab in ((True, "#2a78d6", "spans at full resolution"), (False, "#eb6834", "does not span")):
        m = c.spans_full == flag
        ax.scatter(c.span_frac_full[m], c[f"span_frac_bin{wr}"][m], s=12, color=col, lw=0, alpha=0.8, label=lab)
    ax.plot([0, 1], [0, 1], color=INK, lw=0.8, ls=(0, (4, 3)))
    ax.set_xlabel(f"spanning pore fraction, {FULL_RES}$^3$"); ax.set_ylabel(f"after re-binarisation to {wr}$^3$"); ax.set_title("Connectivity changes when re-binarised")
    ax.legend(fontsize=7.5, loc="upper left")
    ax = axes[2]
    ax.scatter(c.phi, c[f"phi_bin{wr}"] - c.phi, s=12, color="#eb6834", lw=0, alpha=0.8, label=f"re-binarised {wr}$^3$")
    ax.axhline(0, color="#2a78d6", lw=1.6, label="two-channel input (exact)" if POR_ERR < 1e-6 else f"two-channel input (max error {POR_ERR:.1e})")
    ax.set_xlabel("porosity (voxel count)"); ax.set_ylabel("porosity error of the input"); ax.set_title("Porosity is conserved, not predicted")
    ax.legend(fontsize=7.5)
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F03_input_representation")
if RES_V: fig03()

## 9. Main benchmark: accuracy, coverage, width and conditional coverage over seeds

Every number below is computed per property, per seed and per split, and summarised as mean ± standard deviation over seeds.
Conditional coverage is reported per bin with the bin edges, the number of test volumes and the width, using the same bins for every method.
Three binnings: predicted value (the groups Mondrian calibrates on), porosity (a label-free variable no method uses) and the reference value (not observable at deployment).

In [ ]:
# ============================== 9a. Point accuracy, physics and descriptor baselines ==============================
MAIN = [f"main_s{s}" for s in range(cfg.n_seeds)]
rows = []; BASE_PRED = {}
KC_X = np.column_stack([np.log10(DESC[:, 0] + 1e-9), np.log10(DESC[:, 1] + 1e-9), np.log10(DESC[:, 2] + 1e-9)])   # porosity, surface density, mean pore size
for s, tag in enumerate(tqdm(MAIN, desc="baselines")):
    Pm = load_preds(tag); sp = SPLITS[s]
    for j, pn in enumerate(PNAMES):
        rows.append(dict(seed=s, prop=pn, model="3D CNN (this work)", **reg_metrics(Pm["te"]["y"][:, j], Pm["te"]["mu"][:, j])))
        gb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=s).fit(DFEAT_RAW[sp["tr"]], Yt[sp["tr"], j])
        pc, pt = gb.predict(DFEAT_RAW[sp["cal"]]), gb.predict(DFEAT_RAW[sp["te"]])
        rows.append(dict(seed=s, prop=pn, model="Gradient boosting on image descriptors", **reg_metrics(Yt[sp["te"], j], pt)))
        BASE_PRED[(s, pn, "gbm")] = (pc, pt)
        if pn == "log_k":
            lr_ = LinearRegression().fit(KC_X[sp["tr"]], Yt[sp["tr"], j]); pc, pt = lr_.predict(KC_X[sp["cal"]]), lr_.predict(KC_X[sp["te"]])
            rows.append(dict(seed=s, prop=pn, model="Kozeny-Carman-type regression", **reg_metrics(Yt[sp["te"], j], pt)))
            BASE_PRED[(s, pn, "kc")] = (pc, pt)
ACC = pd.DataFrame(rows)
T05 = ACC.groupby(["prop", "model"], sort=False)[["MAE", "RMSE", "R2"]].agg(["mean", "std"]).reset_index()
T05.columns = ["_".join([c for c in col if c]) for col in T05.columns]
save_table(T05, "T05_point_accuracy_by_property")

In [ ]:
# ============================== 9b. Main benchmark over seeds ==============================
ALL_M = ["split", "norm", "cqr", "mondrian", "gauss", "mcd", "mcd_tot"]
ENS_TAGS = [f"gauss_s0_m{m}" for m in range(cfg.ens_M)] if cfg.run_ablations else []
ENS_OK = all(os.path.exists(os.path.join(D["preds"], t + ".npz")) for t in ENS_TAGS) and len(ENS_TAGS) > 1
def bin_edges_for(tr_vals, G=5): return np.quantile(tr_vals, np.linspace(0, 1, G + 1)[1:-1])
main_rows, bin_rows, sweep_rows, info_rows = [], [], [], []
for s, tag in enumerate(tqdm(MAIN, desc="evaluate")):
    Pm = load_preds(tag); cal, te, tr = Pm["cal"], Pm["te"], Pm["tr"]
    for j, pn in enumerate(PNAMES):
        y = te["y"][:, j]
        binvars = {"predicted value": (te["mu"][:, j], bin_edges_for(tr["mu"][:, j])),
                   "porosity": (PHI[te["idx"]], bin_edges_for(PHI[tr["idx"]])),
                   "true value": (y, bin_edges_for(tr["y"][:, j]))}
        for m in ALL_M:
            lo, hi, info = interval(m, cal, te, j, cfg.alpha, tr=tr)
            hit = (y >= lo) & (y <= hi); w = hi - lo
            row = dict(seed=s, prop=pn, method=m, coverage=hit.mean(), width_mean=w.mean(), width_median=np.median(w), n_cal=len(cal["y"]), n_test=len(y))
            for bv, (v, e) in binvars.items():
                g = np.searchsorted(e, v, side="right"); cs = []
                for b in range(len(e) + 1):
                    mk = g == b; n_b = int(mk.sum()); c_b = float(hit[mk].mean()) if n_b else np.nan; cs.append(c_b)
                    bin_rows.append(dict(seed=s, prop=pn, method=m, binvar=bv, bin=b, lo_edge=(-np.inf if b == 0 else e[b - 1]), hi_edge=(np.inf if b == len(e) else e[b]),
                                         n=n_b, coverage=c_b, width=float(w[mk].mean()) if n_b else np.nan))
                row["worst_" + bv.split()[0]] = np.nanmin(cs)
            main_rows.append(row)
            if info: info_rows.append(dict(seed=s, prop=pn, method=m, **info))
        for m in ALL_M + ["cqr_fixed"]:
            for t in cfg.sweep_targets:
                l2, h2, _ = interval(m, cal, te, j, 1 - t, tr=tr)
                sweep_rows.append(dict(seed=s, prop=pn, method=m, target=t, coverage=cov(y, l2, h2), width=float((h2 - l2).mean())))
        for key, name in (("gbm", "split_gbm"), ("kc", "split_kc")):       # the same wrapper on cheaper predictors
            if (s, pn, key) in BASE_PRED:
                pc, pt = BASE_PRED[(s, pn, key)]; q = cq(np.abs(Yt[SPLITS[s]["cal"], j] - pc), cfg.alpha)
                main_rows.append(dict(seed=s, prop=pn, method=name, coverage=cov(Yt[SPLITS[s]["te"], j], pt - q, pt + q), width_mean=2 * q, width_median=2 * q,
                                      n_cal=len(pc), n_test=len(pt)))
    if s == 0 and ENS_OK:
        ec, et = ensemble_pred(ENS_TAGS, "cal"), ensemble_pred(ENS_TAGS, "te")
        for j, pn in enumerate(PNAMES):
            lo, hi, _ = interval("gauss", ec, et, j, cfg.alpha); y = et["y"][:, j]
            main_rows.append(dict(seed=0, prop=pn, method="ens", coverage=cov(y, lo, hi), width_mean=(hi - lo).mean(), width_median=np.median(hi - lo), n_cal=len(ec["y"]), n_test=len(y)))
            for t in cfg.sweep_targets:
                l2, h2, _ = interval("gauss", ec, et, j, 1 - t); sweep_rows.append(dict(seed=0, prop=pn, method="ens", target=t, coverage=cov(y, l2, h2), width=float((h2 - l2).mean())))
R = pd.DataFrame(main_rows); BINS = pd.DataFrame(bin_rows); SWEEP = pd.DataFrame(sweep_rows); INFO = pd.DataFrame(info_rows)
R["abs_err"] = (R.coverage - (1 - cfg.alpha)).abs()
TARGET = 1 - cfg.alpha
N_CAL, N_TEST = int(R.n_cal.iloc[0]), int(R.n_test.iloc[0]); LAW_M, LAW_S = cov_law(N_CAL, N_TEST, cfg.alpha)
print(f"n_cal = {N_CAL}, n_test = {N_TEST}. Exact law for split conformal test coverage: {LAW_M:.4f} +/- {LAW_S:.4f}")

T06 = R.groupby(["prop", "method"], sort=False).agg(coverage_mean=("coverage", "mean"), coverage_sd=("coverage", "std"), abs_err_mean=("abs_err", "mean"),
                                                    width_mean=("width_mean", "mean"), width_sd=("width_mean", "std"), width_median=("width_median", "mean"),
                                                    worst_group_pred=("worst_predicted", "mean"), worst_group_porosity=("worst_porosity", "mean"),
                                                    worst_group_true=("worst_true", "mean"), n_seeds=("seed", "nunique")).reset_index()
T06["method_label"] = T06.method.map(lambda m: MLAB.get(m, {"split_gbm": "Split conformal on gradient boosting", "split_kc": "Split conformal on Kozeny-Carman"}.get(m, m)))
save_table(T06, "T06_coverage_and_width_by_property")

# Summary across properties. The error is |coverage - target| computed per property and per seed, THEN averaged,
# so over- and under-coverage cannot cancel (Reviewer 2, comment 7).
Rm = R[R.method.isin(ALL_M + ["ens"])].copy()
Rm = Rm.merge(R[R.method == "split"][["seed", "prop", "width_mean"]].rename(columns={"width_mean": "w_split"}), on=["seed", "prop"])
Rm["rel_width"] = Rm.width_mean / Rm.w_split
T07 = Rm.groupby("method", sort=False).agg(mean_abs_cov_error_pp=("abs_err", lambda x: 100 * x.mean()),
                                           max_abs_cov_error_pp=("abs_err", lambda x: 100 * x.max()),
                                           mean_coverage=("coverage", "mean"),
                                           mean_worst_group_pred=("worst_predicted", "mean"),
                                           mean_worst_group_porosity=("worst_porosity", "mean"),
                                           width_relative_to_split=("rel_width", "mean")).reset_index()
T07.insert(1, "label", T07.method.map(MLAB)); T07.insert(2, "guarantee", T07.method.map(lambda m: "finite-sample" if m in CONF else "none"))
save_table(T07, "T07_summary_across_properties")

# Paired differences against split conformal, same seed and split (Reviewer 2, comment 8).
rows = []
for pn in PNAMES:
    base = R[(R.method == "split") & (R.prop == pn)].set_index("seed")
    for m in ALL_M[1:]:
        d = R[(R.method == m) & (R.prop == pn)].set_index("seed"); k = base.index.intersection(d.index)
        dc = d.loc[k, "coverage"] - base.loc[k, "coverage"]; rw = d.loc[k, "width_mean"] / base.loc[k, "width_mean"]
        dw = d.loc[k, "worst_porosity"] - base.loc[k, "worst_porosity"]
        pw = st.wilcoxon(rw - 1).pvalue if len(k) >= 6 and np.any(rw != 1) else np.nan
        rows.append(dict(prop=pn, method=m, d_coverage_mean=dc.mean(), d_coverage_sd=dc.std(), width_ratio_mean=rw.mean(), width_ratio_sd=rw.std(),
                         wilcoxon_p_width=pw, d_worst_porosity_group_mean=dw.mean(), d_worst_porosity_group_sd=dw.std(), n_pairs=len(k)))
save_table(pd.DataFrame(rows), "T08_paired_differences_vs_split")
T09 = BINS.groupby(["prop", "binvar", "bin", "method"], sort=False).agg(lo_edge=("lo_edge", "mean"), hi_edge=("hi_edge", "mean"), n_per_seed=("n", "mean"),
                                                                        coverage_mean=("coverage", "mean"), coverage_sd=("coverage", "std"), width_mean=("width", "mean")).reset_index()
save_table(T09, "T09_conditional_coverage_per_bin", show=False)
if len(INFO): save_table(INFO.groupby(["method", "prop"]).mean(numeric_only=True).reset_index().drop(columns="seed"), "T10_cqr_and_mondrian_bookkeeping")
display(T09[(T09.prop == "log_k") & (T09.binvar == "porosity") & (T09.method.isin(CONF))].round(3))

In [ ]:
# ============================== Figure 4: predictive accuracy ==============================
def fig04():
    Pm = load_preds("main_s0")["te"]
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.6)); axes = axes.ravel()
    for j, pn in enumerate(PNAMES):
        ax = axes[j]; y, p = Pm["y"][:, j], Pm["mu"][:, j]
        lim = [min(y.min(), p.min()), max(y.max(), p.max())]
        hb = ax.hexbin(y, p, gridsize=45, cmap=SEQ, mincnt=1, linewidths=0, extent=lim + lim, bins="log")
        ax.plot(lim, lim, color=INK, lw=0.8, ls=(0, (4, 3)))
        a = ACC[(ACC.prop == pn) & (ACC.model.str.startswith("3D CNN"))]
        ax.text(0.04, 0.96, f"$R^2$ = {a.R2.mean():.3f} ± {a.R2.std():.3f}\nRMSE = {a.RMSE.mean():.3f}\nMAE = {a.MAE.mean():.3f}",
                transform=ax.transAxes, va="top", fontsize=8, color=INK)
        ax.set_title(PSHORT[pn]); ax.set_xlabel("reference  " + PLABEL[pn]); ax.set_ylabel("predicted"); ax.set_aspect("equal", "box")
    fig.suptitle(f"Predicted vs reference on held-out test volumes (seed 0 shown; statistics are mean ± sd over {cfg.n_seeds} seeds)", x=0.01, ha="left", fontsize=9.5)
    fig.tight_layout(); savefig(fig, "F04_parity")
fig04()

In [ ]:
# ============================== Figure 5: reliability across target levels ==============================
def fig05():
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.1), sharey=True)
    ts = np.array(cfg.sweep_targets); band = np.array([2 * cov_law(N_CAL, N_TEST, 1 - t)[1] for t in ts]) * 100
    for ax, props, title in ((axes[0], ["log_k"], "log-permeability"), (axes[1], PNAMES, f"mean over {P} properties")):
        ax.fill_between(ts * 100, -band, band, color=GRID, lw=0, label="±2 sd, exact law")
        ax.axhline(0, color=INK, lw=0.8)
        for m in ALL_M + (["ens"] if ENS_OK else []):
            d = SWEEP[(SWEEP.method == m) & (SWEEP.prop.isin(props))]
            g = d.groupby(["seed", "target"]).coverage.mean().reset_index().groupby("target").coverage
            mu, sd = g.mean().values, g.std().fillna(0).values
            ax.plot(ts * 100, (mu - ts) * 100, **mstyle(m))
            if m in CONF: ax.fill_between(ts * 100, (mu - sd - ts) * 100, (mu + sd - ts) * 100, color=MCOL[m], alpha=0.12, lw=0)
        ax.set_title(title); ax.set_xlabel("target coverage (%)")
    axes[0].set_ylabel("empirical − target coverage (percentage points)")
    lim = np.nanpercentile(np.abs((SWEEP.coverage - SWEEP.target) * 100), 99.5); axes[0].set_ylim(-max(lim, 6) * 1.05, max(lim, 6) * 1.05)
    method_legend(fig, ALL_M + (["ens"] if ENS_OK else []), loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.12))
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F05_reliability")
fig05()

In [ ]:
# ============================== Figure 6: coverage at the 90% target, every seed ==============================
def fig06():
    ms = ALL_M + (["ens"] if ENS_OK else [])
    fig, axes = plt.subplots(2, 3, figsize=(11, 6.4), sharey=True); axes = axes.ravel()
    for j, pn in enumerate(PNAMES):
        ax = axes[j]
        ax.axvspan(LAW_M - 2 * LAW_S, LAW_M + 2 * LAW_S, color=GRID, lw=0); ax.axvline(TARGET, color=INK, lw=0.8)
        for i, m in enumerate(ms):
            d = R[(R.prop == pn) & (R.method == m)].coverage.values; yy = len(ms) - 1 - i
            ax.plot(d, np.full(len(d), yy), ls="none", marker="o", ms=2.6, color=MCOL[m], alpha=0.35, mew=0)
            ax.errorbar(d.mean(), yy, xerr=d.std() if len(d) > 1 else 0, color=MCOL[m], capsize=2.5, lw=1.2, **{k: v for k, v in mstyle(m, line=False).items() if k != "color" and k != "linestyle"}, ls="none")
        ax.set_yticks(range(len(ms))); ax.set_yticklabels([MLAB[m] for m in ms[::-1]]); ax.set_title(PSHORT[pn]); ax.grid(axis="y", visible=False)
        if j >= 3: ax.set_xlabel("empirical coverage")
    lo = min(R[R.method.isin(ms)].coverage.min(), 0.8); axes[0].set_xlim(max(lo - 0.02, 0), 1.005)
    fig.suptitle(f"Coverage at the {TARGET:.0%} target: small dots are seeds, symbols are mean ± sd; grey band is ±2 sd of the exact split-conformal law", x=0.01, ha="left", fontsize=9.5)
    fig.tight_layout(); savefig(fig, "F06_coverage_by_seed")
fig06()

In [ ]:
# ============================== Figure 7: interval width, in distribution and in physical terms ==============================
def fig07():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.9), gridspec_kw=dict(width_ratios=[1.15, 1, 1.25]))
    ax = axes[0]; ms = ["norm", "cqr", "mondrian", "gauss", "mcd", "mcd_tot"]
    ax.axhline(1, color=MCOL["split"], lw=1.4); ax.text(P - 0.45, 1.0, "split", color=INK2, va="bottom", ha="right", fontsize=7.5)
    for i, m in enumerate(ms):
        d = Rm[Rm.method == m].groupby("prop", sort=False).rel_width.agg(["mean", "std"]).reindex(PNAMES)
        ax.errorbar(np.arange(P) + (i - 2.5) * 0.11, d["mean"], yerr=d["std"].fillna(0), capsize=1.5, lw=0.9, ls="none",
                    **{k: v for k, v in mstyle(m, line=False).items() if k != "linestyle"})
    prop_ticks(ax)
    ax.set_ylabel("mean width relative to split conformal"); ax.set_title("Interval width"); ax.set_yscale("log")
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}")); ax.yaxis.set_minor_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    # (b) distribution of the multiplicative uncertainty on permeability
    ax = axes[1]; Pm = load_preds("main_s0")
    for m in ["split", "norm", "cqr", "mondrian"]:
        lo, hi, _ = interval(m, Pm["cal"], Pm["te"], J_K, cfg.alpha, tr=Pm["tr"]); f = np.sort(10 ** ((hi - lo) / 2))
        ax.plot(f, np.linspace(0, 1, len(f)), color=MCOL[m], ls=METHODS[m][3], lw=1.6, label=MLAB[m])
    ax.set_xlabel("multiplicative uncertainty on $k$  (×/÷ factor)"); ax.set_ylabel("fraction of test volumes"); ax.set_title("Width distribution, permeability")
    ax.legend(loc="lower right")
    # (c) intervals in physical units for a random sample of rocks
    ax = axes[2]; te = Pm["te"]; lo, hi, _ = interval("cqr", Pm["cal"], te, J_K, cfg.alpha, tr=Pm["tr"])
    ii = np.random.default_rng(0).choice(len(lo), min(60, len(lo)), replace=False); ii = ii[np.argsort(te["mu"][ii, J_K])]
    y = te["y"][ii, J_K]; hit = (y >= lo[ii]) & (y <= hi[ii]); x = np.arange(len(ii))
    ax.vlines(x, 10 ** lo[ii], 10 ** hi[ii], color=MCOL["cqr"], lw=1.6, alpha=0.9, label="90% CQR interval")
    ax.plot(x[hit], 10 ** y[hit], ls="none", marker="o", ms=3, color=INK, label="reference value, inside")
    ax.plot(x[~hit], 10 ** y[~hit], ls="none", marker="x", ms=5.5, mew=1.4, color=INK, label="reference value, outside")
    ax.set_yscale("log"); ax.set_xlabel("test volumes, sorted by predicted permeability"); ax.set_ylabel("$k$  [px$^2$]"); ax.set_title("Intervals in physical units")
    ax.legend(loc="upper left", fontsize=7.5); ax.set_xticks([])
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F07_interval_width")
fig07()

# Table: what the intervals mean for permeability (Reviewer 2, comment 11).
rows = []
for s, tag in enumerate(MAIN):
    Pm = load_preds(tag)
    for m in ["split", "norm", "cqr", "mondrian"]:
        lo, hi, _ = interval(m, Pm["cal"], Pm["te"], J_K, cfg.alpha, tr=Pm["tr"]); hw = (hi - lo) / 2
        rows.append(dict(seed=s, method=MLAB[m], median_half_width_decades=np.median(hw), median_fold_factor=10 ** np.median(hw),
                         p10_fold_factor=10 ** np.percentile(hw, 10), p90_fold_factor=10 ** np.percentile(hw, 90)))
T11 = pd.DataFrame(rows).groupby("method", sort=False).mean(numeric_only=True).drop(columns="seed").reset_index()
save_table(T11, "T11_permeability_interval_in_physical_terms")
PX2_TO_MD = cfg.voxel_um**2 * 1013.25
print(f"Units: DeePore lengths are voxels of the {FULL_RES}^3 grid, so k[m^2] = k[px^2] x (voxel size)^2. "
      f"Example: at {cfg.voxel_um} um per voxel, 1 px^2 = {PX2_TO_MD:,.0f} mD. An interval [lo, hi] in log10 k maps to [10^lo, 10^hi] x that factor.")

In [ ]:
# ============================== Figure 8: coverage within groups ==============================
def fig08():
    bvs = ["predicted value", "porosity", "true value"]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.9), sharey=True)
    for ax, bv in zip(axes, bvs):
        d0 = BINS[(BINS.prop == "log_k") & (BINS.binvar == bv)]
        nb = int(d0.n.mean()); sd_b = np.sqrt(TARGET * (1 - TARGET) / max(nb, 1))
        ax.axhspan(TARGET - 2 * sd_b, TARGET + 2 * sd_b, color=GRID, lw=0); ax.axhline(TARGET, color=INK, lw=0.8)
        for i, m in enumerate(CONF):
            g = d0[d0.method == m].groupby("bin").coverage; x = np.arange(len(g)) + (i - 1.5) * 0.08
            ax.errorbar(x, g.mean(), yerr=g.std().fillna(0), capsize=2, lw=1.1, **mstyle(m))
        ax.set_xticks(range(5)); ax.set_xticklabels(["lowest", "2", "3", "4", "highest"]); ax.set_xlabel(f"quintile of {bv}")
        ax.set_title({"predicted value": "Groups used by Mondrian", "porosity": "Groups not used by any method", "true value": "Groups by reference value"}[bv])
    axes[0].set_ylabel(f"coverage of log-permeability ({TARGET:.0%} target)")
    method_legend(fig, CONF, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.1))
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F08_conditional_coverage")
fig08()

## 10. Is the implementation valid? Calibration size and hyper-parameter sensitivity (Reviewer 2, comments 3, 5, 6, 8, 14)

**An exact test of the implementation.** Hold one trained predictor fixed and pool its calibration and test volumes. For a random partition of a fixed pool,
the ranks of the scores are uniformly random, so the number of covered test points follows a **beta-binomial law exactly**: for $n$ calibration and $m$ test points,
$H\sim\text{BetaBinomial}(m,\ k,\ n+1-k)$ with $k=\lceil (n+1)(1-\alpha)\rceil$. This holds for split, normalized and CQR scores alike (they do not use the calibration data).
A wrong quantile convention, leakage between calibration and test, or a score that peeks at calibration data shifts this distribution. We compare the full distribution,
not only its mean.

**Calibration size.** All sizes are drawn from the same pool of `cal ∪ test` volumes of seed 0..2, which the predictor never saw. The test size is fixed at 1,000 (or the remainder),
so the curves differ only through $n$. Sample counts for every setting are written to `T14`.

**Sensitivity.** The normalized-score constant $\varepsilon$, the number of Mondrian groups, how Mondrian groups are formed, and the quantile pair used by CQR.

In [ ]:
# ============================== 10a. Repartition analysis on real predictions ==============================
REP_SEEDS = list(range(min(3, cfg.n_seeds)))
def merge_parts(Pm, parts=("cal", "te")):
    out = {}
    for k in ["y", "mu", "sig", "q", "idx"]: out[k] = np.concatenate([Pm[p][k] for p in parts])
    return out
def take(d, ii): return {k: v[ii] for k, v in d.items()}
rep_rows, size_rows = [], []
for s in REP_SEEDS:
    Pm = load_preds(f"main_s{s}"); pool = merge_parts(Pm); npool = len(pool["y"]); rng = np.random.default_rng(500 + s)
    m_test = min(1000, npool // 3)
    grid = [n for n in [25, 50, 100, 200, 400, 800, 1600, 3200, 6400] if n <= npool - m_test]
    size_rows += [dict(seed=s, pool_size=npool, n_cal=n, n_test=m_test, n_repartitions=cfg.n_repart, train_size=len(Pm["tr"]["y"]),
                       disjoint_from_training=bool(len(np.intersect1d(pool["idx"], Pm["tr"]["idx"])) == 0)) for n in grid]
    for n in grid:
        for r in range(cfg.n_repart):
            p = rng.permutation(npool); c_, t_ = take(pool, p[:n]), take(pool, p[n:n + m_test])
            for j in (J_K, 1):
                for m in ["split", "norm", "cqr", "mondrian"]:
                    lo, hi, _ = interval(m, c_, t_, j, cfg.alpha, tr=Pm["tr"])
                    rep_rows.append(dict(seed=s, n_cal=n, rep=r, prop=PNAMES[j], method=m, hits=int(((t_["y"][:, j] >= lo) & (t_["y"][:, j] <= hi)).sum()), m=m_test))
REP = pd.DataFrame(rep_rows); REP["coverage"] = REP.hits / REP.m
save_table(pd.DataFrame(size_rows), "T14_calibration_size_sample_counts")
def discrete_ks(h, law, mt, n_mc=2000, seed=0):
    """KS distance between the empirical and the exact discrete CDF, evaluated on the support (right-continuous),
    with a Monte-Carlo p-value from the same law (scipy's kstest assumes a continuous law and is invalid here)."""
    sup = np.arange(mt + 1); F = law.cdf(sup)
    def D(x): return np.abs(np.searchsorted(np.sort(x), sup, side="right") / len(x) - F).max()
    d0 = D(h); sims = law.rvs(size=(n_mc, len(h)), random_state=seed)
    return float(d0), float((1 + sum(D(r) >= d0 for r in sims)) / (n_mc + 1))
rows = []
for (n, pn, m), d in REP.groupby(["n_cal", "prop", "method"]):
    k = int(np.ceil((n + 1) * (1 - cfg.alpha) - 1e-9)); mt = int(d.m.iloc[0])
    if k > n: continue
    law = st.betabinom(mt, k, n + 1 - k)
    D_, p_ = discrete_ks(d.hits.values, law, mt)
    rows.append(dict(n_cal=n, prop=pn, method=m, n_draws=len(d), cov_mean=d.coverage.mean(), cov_sd=d.coverage.std(),
                     law_mean=law.mean() / mt, law_sd=law.std() / mt, ks_stat=D_, ks_p=p_))
VALID = pd.DataFrame(rows); save_table(VALID[VALID.method != "mondrian"], "T15_coverage_distribution_vs_exact_law")
print("Mondrian is excluded from the exact-law test: its marginal law is a mixture over groups (it is exact within each group).")

In [ ]:
# ============================== 10b. Sensitivity: epsilon, Mondrian groups and grouping rule, CQR quantile pair ==============================
eps_rows, grp_rows, edge_rows = [], [], []
EPS_GRID = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
G_GRID = [1, 2, 3, 5, 8, 10, 15, 20]
for s, tag in enumerate(MAIN):
    Pm = load_preds(tag); cal, te, tr = Pm["cal"], Pm["te"], Pm["tr"]
    for j, pn in enumerate(PNAMES):
        y = te["y"][:, j]; pe = bin_edges_for(PHI[tr["idx"]]); gp = np.searchsorted(pe, PHI[te["idx"]], side="right")
        for e in EPS_GRID:
            lo, hi, _ = interval("norm", cal, te, j, cfg.alpha, tr=tr, eps_rel=e)
            eps_rows.append(dict(seed=s, prop=pn, eps_rel=e, coverage=cov(y, lo, hi), width=(hi - lo).mean()))
        for G in G_GRID:
            lo, hi, info = interval("mondrian", cal, te, j, cfg.alpha, tr=tr, G=G); hit = (y >= lo) & (y <= hi)
            own = np.searchsorted(group_edges(tr["mu"][:, j], G), te["mu"][:, j], side="right")
            grp_rows.append(dict(seed=s, prop=pn, G=G, coverage=hit.mean(), width=(hi - lo).mean(),
                                 worst_own_group=min(hit[own == g].mean() for g in range(G) if (own == g).any()),
                                 worst_porosity_quintile=min(hit[gp == g].mean() for g in range(5) if (gp == g).any()),
                                 min_cal_group_n=info.get("min_group_n", len(cal["y"])), fallback_groups=info.get("n_fallback_groups", 0)))
        # grouping rule: edges from training predictions (used), from the calibration predictions themselves (original submission), equal-width on training range
        res = np.abs(cal["y"][:, j] - cal["mu"][:, j])
        for rule, e in (("training-prediction quintiles (used)", group_edges(tr["mu"][:, j], 5)),
                        ("calibration-prediction quintiles (original)", group_edges(cal["mu"][:, j], 5)),
                        ("equal-width on training range", np.linspace(tr["mu"][:, j].min(), tr["mu"][:, j].max(), 6)[1:-1])):
            gc_, gt_ = np.searchsorted(e, cal["mu"][:, j], side="right"), np.searchsorted(e, te["mu"][:, j], side="right")
            qg = np.array([cq(res[gc_ == g], cfg.alpha) if (gc_ == g).sum() else np.inf for g in range(5)]); qg[~np.isfinite(qg)] = cq(res, cfg.alpha)
            lo, hi = te["mu"][:, j] - qg[gt_], te["mu"][:, j] + qg[gt_]; hit = (y >= lo) & (y <= hi)
            edge_rows.append(dict(seed=s, prop=pn, rule=rule, coverage=hit.mean(), width=(hi - lo).mean(),
                                  worst_group=min(hit[gt_ == g].mean() for g in range(5) if (gt_ == g).any())))
EPS = pd.DataFrame(eps_rows); GRP = pd.DataFrame(grp_rows); EDGE = pd.DataFrame(edge_rows)
save_table(EPS.groupby(["prop", "eps_rel"]).agg(coverage=("coverage", "mean"), coverage_sd=("coverage", "std"), width=("width", "mean")).reset_index(), "T16_epsilon_sensitivity", show=False)
save_table(GRP.groupby(["prop", "G"]).mean(numeric_only=True).drop(columns="seed").reset_index(), "T17_mondrian_group_count", show=False)
save_table(EDGE.groupby(["rule"], sort=False).agg(coverage=("coverage", "mean"), abs_err=("coverage", lambda x: (x - TARGET).abs().mean()),
                                                   worst_group=("worst_group", "mean"), width=("width", "mean")).reset_index(), "T18_mondrian_grouping_rule")
CQRF = SWEEP[SWEEP.method.isin(["cqr", "cqr_fixed"])].pivot_table(index=["seed", "prop", "target"], columns="method", values=["coverage", "width"]).reset_index()
CQRF.columns = ["_".join([c for c in col if c]) for col in CQRF.columns]
CQRF["width_ratio_fixed_over_matched"] = CQRF.width_cqr_fixed / CQRF.width_cqr
save_table(CQRF.groupby("target").mean(numeric_only=True).drop(columns="seed").reset_index(), "T19_cqr_quantile_pair")

In [ ]:
# ============================== Figure 9: validity of the implementation ==============================
def fig09():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.7))
    ax = axes[0]; n0 = max(n for n in VALID.n_cal.unique() if n <= N_CAL)
    d = REP[(REP.n_cal == n0) & (REP.prop == "log_k") & (REP.method == "split")]; mt = int(d.m.iloc[0])
    k = int(np.ceil((n0 + 1) * (1 - cfg.alpha) - 1e-9)); law = st.betabinom(mt, k, n0 + 1 - k)
    h = np.arange(law.ppf(0.0005), law.ppf(0.9995) + 1); bw = max(1, int(len(h) / 30))
    ax.hist(d.hits / mt, bins=np.arange(h[0], h[-1] + bw, bw) / mt, density=True, color="#b7d3f6", edgecolor=SURF, lw=0.4, label=f"{len(d)} repartitions")
    ax.plot(h / mt, law.pmf(h) * mt, color=INK, lw=1.4, label="exact beta-binomial law")
    v = VALID[(VALID.n_cal == n0) & (VALID.prop == "log_k") & (VALID.method == "split")].iloc[0]
    ax.set_title(f"Split conformal, n = {n0}, m = {mt}"); ax.set_xlabel("test coverage"); ax.set_ylabel("density"); ax.legend(fontsize=7.5, loc="upper left")
    ax.text(0.98, 0.95, f"discrete KS p = {v.ks_p:.2f}", transform=ax.transAxes, ha="right", va="top", fontsize=8)
    for ax, col, ttl in ((axes[1], "cov_mean", "Mean coverage vs calibration size"), (axes[2], "cov_sd", "Spread of coverage vs calibration size")):
        lawc = VALID[(VALID.prop == "log_k") & (VALID.method == "split")].sort_values("n_cal")
        if col == "cov_mean":
            ax.fill_between(lawc.n_cal, lawc.law_mean - 2 * lawc.law_sd, lawc.law_mean + 2 * lawc.law_sd, color=GRID, lw=0, label="exact law, ±2 sd")
            ax.plot(lawc.n_cal, lawc.law_mean, color=INK, lw=0.9); ax.axhline(TARGET, color=INK2, lw=0.6, ls=(0, (4, 3)))
        else:
            ax.plot(lawc.n_cal, lawc.law_sd, color=INK, lw=1.2, label="exact law")
        for m in ["split", "norm", "cqr", "mondrian"]:
            dd = REP[(REP.prop == "log_k") & (REP.method == m)].groupby("n_cal").coverage
            ax.plot(dd.mean().index if col == "cov_mean" else dd.std().index, dd.mean().values if col == "cov_mean" else dd.std().values, **mstyle(m))
        ax.set_xscale("log"); ax.set_xlabel("calibration size n"); ax.set_title(ttl)
        ns = sorted(REP.n_cal.unique()); ax.set_xticks(ns); ax.set_xticklabels([str(v) for v in ns], fontsize=7.5); ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
        ax.set_ylabel("coverage of log-permeability" if col == "cov_mean" else "sd of test coverage")
    axes[1].legend(fontsize=7.5, loc="lower right"); method_legend(fig, ["split", "norm", "cqr", "mondrian"], loc="lower center", ncol=4, bbox_to_anchor=(0.62, -0.1))
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F09_validity_and_calibration_size")
fig09()

In [ ]:
# ============================== Figure 10: sensitivity ==============================
def fig10():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.7))
    ax = axes[0]; ax.axhline(TARGET, color=INK, lw=0.8); ax.axvline(cfg.eps_rel, color=GRID, lw=4, zorder=0)
    for pn, mk in (("log_k", "s"), ("log_F", "o")):
        d = EPS[EPS.prop == pn].groupby("eps_rel"); w = d.width.mean(); dw = 100 * (w / w.loc[cfg.eps_rel] - 1)
        ax.errorbar(d.coverage.mean().index, d.coverage.mean(), yerr=d.coverage.std(), color=MCOL["norm"], marker=mk, ms=5,
                    mfc=MCOL["norm"] if pn == "log_k" else SURF, capsize=2, ls="-" if pn == "log_k" else (0, (4, 2)),
                    label=f"{PSHORT[pn]} (width change {dw.min():+.0f}% to {dw.max():+.0f}%)")
    ax.set_xscale("log"); ax.set_xlabel(r"$\varepsilon$ / training sd of the target"); ax.set_ylabel("coverage")
    ax.set_title("Normalized conformal: $\\varepsilon$"); ax.legend(fontsize=7, loc="lower left")
    ax = axes[1]; d = GRP[GRP.prop == "log_k"].groupby("G")
    ax.axhline(TARGET, color=INK, lw=0.8)
    ax.errorbar(d.coverage.mean().index, d.coverage.mean(), yerr=d.coverage.std(), color=MCOL["mondrian"], marker="D", ms=5, capsize=2, label="marginal")
    ax.errorbar(d.worst_own_group.mean().index, d.worst_own_group.mean(), yerr=d.worst_own_group.std(), color=MCOL["mondrian"], marker="D", ms=5, mfc=SURF, ls=(0, (4, 2)), capsize=2, label="worst calibrated group")
    ax.errorbar(d.worst_porosity_quintile.mean().index, d.worst_porosity_quintile.mean(), yerr=d.worst_porosity_quintile.std(), color=INK2, marker="o", ms=4, mfc=SURF, ls=(0, (1, 2)), capsize=2, label="worst porosity quintile")
    ax.set_xlabel("number of Mondrian groups (1 = split)"); ax.set_ylabel("coverage of log-permeability"); ax.set_title("Mondrian: number of groups"); ax.legend(fontsize=7.5, loc="lower left")
    ax = axes[2]; d = CQRF.groupby("target")
    ax.plot(d.width_ratio_fixed_over_matched.mean().index * 100, d.width_ratio_fixed_over_matched.mean().values, color=MCOL["cqr"], marker="^", label="width, fixed / matched")
    ax.axhline(1, color=INK, lw=0.8); ax.set_xlabel("target coverage (%)"); ax.set_ylabel("width ratio"); ax.set_title("CQR: fixed 0.05/0.95 heads vs matched pair")
    ax.legend(fontsize=7.5, loc="upper center")
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F10_sensitivity")
fig10()

## 11. Heuristic baselines and training ablations (Reviewer 2, comments 9 and 12)

- **MC-dropout in two forms.** "Means" uses only the spread of the stochastic means (epistemic part). "Total" adds the mean predicted variance from the Gaussian head,
  $\hat\sigma^2_{tot}=\overline{\hat\sigma^2_t}+\mathrm{Var}_t(\hat\mu_t)$. Convergence is checked up to 100 passes.
- **Dedicated Gaussian network** trained only on the Gaussian likelihood, versus the Gaussian head of the jointly trained network. This tests whether the
  over-coverage of the Gaussian head is caused by the joint training design.
- **Deep ensemble** of five dedicated Gaussian networks (mixture mean and variance).
- **Mean-only network** versus the jointly trained network, same split and same initial seed: do the auxiliary losses change the point predictor?
- **Conformalized heuristics.** Any heuristic scale can be used as the denominator of the normalized score. This turns MC-dropout or ensemble spreads into intervals with a coverage guarantee.

Conclusions in this section apply to these implementations only.

In [ ]:
# ============================== 11a. MC-dropout convergence, conformalized heuristics, ablations ==============================
def norm_with_scale(cal_mu, cal_sc, cal_y, te_mu, te_sc, alpha, eps):
    q = cq(np.abs(cal_y - cal_mu) / (cal_sc + eps), alpha); w = q * (te_sc + eps); return te_mu - w, te_mu + w
Pm0 = load_preds("main_s0"); te0 = Pm0["te"]
mct_rows = []
for T in cfg.mc_T_grid:
    if f"mc_vm_T{T}" not in te0: continue
    for j, pn in enumerate(PNAMES):
        y = te0["y"][:, j]; mu = te0[f"mc_mu_T{T}"][:, j]
        for name, sd in (("mcd", np.sqrt(te0[f"mc_vm_T{T}"][:, j])), ("mcd_tot", np.sqrt(te0[f"mc_vm_T{T}"][:, j] + te0[f"mc_mv_T{T}"][:, j]))):
            w = zq(cfg.alpha) * sd; mct_rows.append(dict(T=T, prop=pn, method=name, coverage=cov(y, mu - w, mu + w), width=2 * w.mean()))
MCT = pd.DataFrame(mct_rows); save_table(MCT.groupby(["method", "T"]).agg(coverage=("coverage", "mean"), width=("width", "mean")).reset_index(), "T20_mc_dropout_convergence")

h_rows = []
for s, tag in enumerate(MAIN):
    Pm = load_preds(tag); cal, te, tr = Pm["cal"], Pm["te"], Pm["tr"]
    for j, pn in enumerate(PNAMES):
        y = te["y"][:, j]; eps = cfg.eps_rel * tr["y"][:, j].std()
        for m in ["gauss", "mcd", "mcd_tot"]:
            lo, hi, _ = interval(m, cal, te, j, cfg.alpha, tr=tr); h_rows.append(dict(seed=s, prop=pn, family=m, version="heuristic", coverage=cov(y, lo, hi), width=(hi - lo).mean()))
        lo, hi = norm_with_scale(cal["mu"][:, j], cal["sig"][:, j], cal["y"][:, j], te["mu"][:, j], te["sig"][:, j], cfg.alpha, eps)
        h_rows.append(dict(seed=s, prop=pn, family="gauss", version="conformalized", coverage=cov(y, lo, hi), width=(hi - lo).mean()))
        for m, sc in (("mcd", lambda d: np.sqrt(d["mc_vm"][:, j])), ("mcd_tot", lambda d: np.sqrt(d["mc_vm"][:, j] + d["mc_mv"][:, j]))):
            lo, hi = norm_with_scale(cal["mc_mu"][:, j], sc(cal), cal["y"][:, j], te["mc_mu"][:, j], sc(te), cfg.alpha, eps)
            h_rows.append(dict(seed=s, prop=pn, family=m, version="conformalized", coverage=cov(y, lo, hi), width=(hi - lo).mean()))
if ENS_OK:
    ec, et = ensemble_pred(ENS_TAGS, "cal"), ensemble_pred(ENS_TAGS, "te"); trs = load_preds(ENS_TAGS[0])["tr"]
    for j, pn in enumerate(PNAMES):
        y = et["y"][:, j]; lo, hi, _ = interval("gauss", ec, et, j, cfg.alpha)
        h_rows.append(dict(seed=0, prop=pn, family="ens", version="heuristic", coverage=cov(y, lo, hi), width=(hi - lo).mean()))
        lo, hi = norm_with_scale(ec["mu"][:, j], ec["sig"][:, j], ec["y"][:, j], et["mu"][:, j], et["sig"][:, j], cfg.alpha, cfg.eps_rel * trs["y"][:, j].std())
        h_rows.append(dict(seed=0, prop=pn, family="ens", version="conformalized", coverage=cov(y, lo, hi), width=(hi - lo).mean()))
HEU = pd.DataFrame(h_rows); HEU["abs_err"] = (HEU.coverage - TARGET).abs()
save_table(HEU.groupby(["family", "version"], sort=False).agg(coverage=("coverage", "mean"), mean_abs_err_pp=("abs_err", lambda x: 100 * x.mean()),
                                                              max_abs_err_pp=("abs_err", lambda x: 100 * x.max()), width=("width", "mean")).reset_index(),
           "T21_heuristics_and_their_conformalized_versions")

abl_rows = []
for s in range(min(cfg.n_abl, cfg.n_seeds)):
    if not cfg.run_ablations: break
    Pj, Pmo, Pg = load_preds(f"main_s{s}"), load_preds(f"meanonly_s{s}"), load_preds(f"gauss_s{s}_m0")
    for j, pn in enumerate(PNAMES):
        y = Pj["te"]["y"][:, j]
        mj, mo = reg_metrics(y, Pj["te"]["mu"][:, j]), reg_metrics(y, Pmo["te"]["mu"][:, j])
        qj = cq(np.abs(Pj["cal"]["y"][:, j] - Pj["cal"]["mu"][:, j]), cfg.alpha); qo = cq(np.abs(Pmo["cal"]["y"][:, j] - Pmo["cal"]["mu"][:, j]), cfg.alpha)
        lo, hi, _ = interval("gauss", Pj["cal"], Pj["te"], j, cfg.alpha); lo2, hi2, _ = interval("gauss", Pg["cal"], Pg["te"], j, cfg.alpha)
        abl_rows.append(dict(seed=s, prop=pn, rmse_joint=mj["RMSE"], rmse_mean_only=mo["RMSE"], rmse_ratio=mj["RMSE"] / mo["RMSE"],
                             split_width_ratio=qj / qo, gauss_cov_joint_head=cov(y, lo, hi), gauss_cov_dedicated=cov(y, lo2, hi2),
                             gauss_width_joint_head=(hi - lo).mean(), gauss_width_dedicated=(hi2 - lo2).mean(),
                             rmse_dedicated_gauss=reg_metrics(y, Pg["te"]["mu"][:, j])["RMSE"]))
ABL = pd.DataFrame(abl_rows)
if len(ABL): save_table(ABL.groupby("prop", sort=False).mean(numeric_only=True).drop(columns="seed").reset_index(), "T22_training_ablation")

In [ ]:
# ============================== Figure 11: heuristics ==============================
def fig11():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.8), gridspec_kw=dict(width_ratios=[1, 1.25, 1]))
    ax = axes[0]; ax.axhline(TARGET, color=INK, lw=0.8)
    for m in ["mcd", "mcd_tot"]:
        d = MCT[MCT.method == m].groupby("T"); ax.plot(d.coverage.mean().index, d.coverage.mean().values, **mstyle(m), label=MLAB[m])
    ax.set_xscale("log"); ax.set_xlabel("number of stochastic passes T"); ax.set_ylabel(f"coverage at {TARGET:.0%} target (mean over properties)")
    ax.set_title("MC-dropout: convergence in T"); ax.legend(fontsize=7.5)
    Ts = sorted(MCT["T"].unique()); ax.set_xticks(Ts); ax.set_xticklabels([str(t) for t in Ts]); ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax = axes[1]; fams = [f for f in ["gauss", "mcd", "mcd_tot", "ens"] if f in HEU.family.unique()]
    ax.axvline(TARGET, color=INK, lw=0.8)
    for i, f in enumerate(fams):
        for k, pn in enumerate(PNAMES):
            d = HEU[(HEU.family == f) & (HEU.prop == pn)].groupby("version").coverage.mean()
            yy = i * (P + 1.5) + k
            if {"heuristic", "conformalized"} <= set(d.index):
                ax.plot([d["heuristic"], d["conformalized"]], [yy, yy], color=GRID, lw=2, zorder=1)
                ax.plot(d["heuristic"], yy, ls="none", marker=METHODS[f][2], ms=6, mfc=SURF, mec=MCOL[f], mew=1.3, zorder=2)
                ax.plot(d["conformalized"], yy, ls="none", marker=METHODS[f][2], ms=6, color=MCOL[f], zorder=3)
    ax.set_yticks([i * (P + 1.5) + k for i in range(len(fams)) for k in range(P)])
    ax.set_yticklabels([f"{MLAB[f].split(' (')[0]}: {PSHORT[pn]}" if k == 0 else PSHORT[pn] for f in fams for k, pn in enumerate(PNAMES)], fontsize=6.5)
    ax.invert_yaxis(); ax.grid(axis="y", visible=False); ax.set_xlabel("coverage")
    ax.set_title("Open: heuristic; filled: conformalized")
    ax = axes[2]
    if len(ABL):
        d = ABL.groupby("prop", sort=False)[["rmse_ratio", "split_width_ratio"]].agg(["mean", "std"]).reindex(PNAMES)
        x = np.arange(P)
        ax.errorbar(x - 0.1, d[("rmse_ratio", "mean")], yerr=d[("rmse_ratio", "std")].fillna(0), ls="none", marker="o", color=MCOL["split"], capsize=2, label="test RMSE")
        ax.errorbar(x + 0.1, d[("split_width_ratio", "mean")], yerr=d[("split_width_ratio", "std")].fillna(0), ls="none", marker="s", mfc=SURF, color=MCOL["split"], capsize=2, label="split-conformal width")
        ax.axhline(1, color=INK, lw=0.8); prop_ticks(ax)
        ax.set_ylabel("joint training / mean-only training"); ax.set_title("Do the auxiliary heads change the predictor?"); ax.legend(fontsize=7.5)
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F11_heuristics_and_ablations")
fig11()

## 12. Joint coverage over several properties (Reviewer 2, comment 4)

Property-wise intervals at 90% do **not** give 90% probability that all properties are covered at once. We measure the joint coverage of independent property-wise
intervals and compare two constructions that do carry a joint guarantee $P(y_j\in C_j(x)\ \forall j)\ge 1-\alpha$:

- **Bonferroni**: property-wise split conformal at level $\alpha/P$.
- **Max-score conformal**: one score per volume, $s=\max_j |y_j-\hat\mu_j(x)|/d_j(x)$, calibrated once; the box is $\hat\mu_j(x)\pm\hat q\,d_j(x)$.
  With $d_j$ = training-set residual sd ("max-score, fixed scale") or $d_j=\hat\sigma_j(x)+\varepsilon_j$ ("max-score, adaptive").
  The scales come from training data only, so the guarantee is exact.

Two property sets: permeability with formation factor (the two transport properties a flow-and-resistivity workflow uses together), and all six.

In [ ]:
# ============================== 12a. Joint coverage ==============================
SETS = {"k + F": [PNAMES.index("log_k"), PNAMES.index("log_F")], f"all {P}": list(range(P))}
JM = {"indep": "Independent 90% intervals", "bonf": "Bonferroni", "max_fixed": "Max-score, fixed scale", "max_adapt": "Max-score, adaptive scale"}
j_rows = []
for s, tag in enumerate(MAIN):
    Pm = load_preds(tag); cal, te, tr = Pm["cal"], Pm["te"], Pm["tr"]
    rsd = (tr["y"] - tr["mu"]).std(0); eps = cfg.eps_rel * tr["y"].std(0)
    for sname, J in SETS.items():
        k = len(J); yt, yc = te["y"][:, J], cal["y"][:, J]
        marg = np.array([2 * cq(np.abs(yc[:, a] - cal["mu"][:, J[a]]), cfg.alpha) for a in range(k)])
        margn = []
        for a in range(k):
            lo, hi, _ = interval("norm", cal, te, J[a], cfg.alpha, tr=tr); margn.append((hi - lo).mean())
        margn = np.array(margn)
        res = {}
        hw = np.array([cq(np.abs(yc[:, a] - cal["mu"][:, J[a]]), cfg.alpha) for a in range(k)])
        res["indep"] = (np.all(np.abs(yt - te["mu"][:, J]) <= hw, 1), 2 * hw, marg)
        hw = np.array([cq(np.abs(yc[:, a] - cal["mu"][:, J[a]]), cfg.alpha / k) for a in range(k)])
        res["bonf"] = (np.all(np.abs(yt - te["mu"][:, J]) <= hw, 1), 2 * hw, marg)
        d = rsd[J]; q = cq(np.max(np.abs(yc - cal["mu"][:, J]) / d, 1), cfg.alpha)
        res["max_fixed"] = (np.all(np.abs(yt - te["mu"][:, J]) <= q * d, 1), 2 * q * d, marg)
        dc, dt = cal["sig"][:, J] + eps[J], te["sig"][:, J] + eps[J]; q = cq(np.max(np.abs(yc - cal["mu"][:, J]) / dc, 1), cfg.alpha)
        res["max_adapt"] = (np.all(np.abs(yt - te["mu"][:, J]) <= q * dt, 1), (2 * q * dt).mean(0), margn)
        for m, (hit, w, wm) in res.items():
            for a in range(k):
                j_rows.append(dict(seed=s, set=sname, method=m, joint_coverage=hit.mean(), prop=PNAMES[J[a]], width=w[a], width_vs_marginal=w[a] / wm[a]))
JOINT = pd.DataFrame(j_rows)
T23 = JOINT.groupby(["set", "method"], sort=False).agg(joint_coverage=("joint_coverage", "mean"), joint_coverage_sd=("joint_coverage", "std"),
                                                        width_vs_marginal=("width_vs_marginal", "mean")).reset_index()
T23["method"] = T23.method.map(JM); save_table(T23, "T23_joint_coverage")

def fig12():
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6), gridspec_kw=dict(width_ratios=[1, 1.3]))
    jcol = {"indep": "#2a78d6", "bonf": "#eb6834", "max_fixed": "#1baf7a", "max_adapt": "#6250d6"}; jmk = {"indep": "o", "bonf": "s", "max_fixed": "^", "max_adapt": "D"}
    ax = axes[0]; ax.axvline(TARGET, color=INK, lw=0.8)
    for i, sname in enumerate(SETS):
        for k_, m in enumerate(JM):
            d = JOINT[(JOINT.set == sname) & (JOINT.method == m)].groupby("seed").joint_coverage.first()
            yy = i * 5 + k_
            ax.errorbar(d.mean(), yy, xerr=d.std() if len(d) > 1 else 0, color=jcol[m], marker=jmk[m], ms=6, capsize=2, ls="none", label=JM[m] if i == 0 else None)
    ax.set_yticks([i * 5 + 1.5 for i in range(len(SETS))]); ax.set_yticklabels(list(SETS)); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set_xlabel("probability that every property is covered"); ax.set_title("Joint coverage"); ax.legend(fontsize=7.5, loc="lower left")
    ax = axes[1]; sname = f"all {P}"
    for k_, m in enumerate(["bonf", "max_fixed", "max_adapt"]):
        d = JOINT[(JOINT.set == sname) & (JOINT.method == m)].groupby("prop", sort=False).width_vs_marginal.agg(["mean", "std"]).reindex(PNAMES)
        ax.errorbar(np.arange(P) + (k_ - 1) * 0.15, d["mean"], yerr=d["std"].fillna(0), color=jcol[m], marker=jmk[m], ms=6, capsize=2, ls="none", label=JM[m])
    ax.axhline(1, color=INK, lw=0.8); prop_ticks(ax)
    ax.set_ylabel("width / property-wise 90% width"); ax.set_title(f"Price of a joint guarantee ({P} properties)"); ax.legend(fontsize=7.5)
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F12_joint_coverage")
fig12()

## 13. Distribution shift (Reviewer 1; Reviewer 2, comments 2 and 10)

Conformal coverage is guaranteed only when calibration and deployment data are exchangeable. Three shift experiments, each with a network **retrained without the withheld data**:

| Experiment | What is withheld | Kind of shift |
|---|---|---|
| Label-support extrapolation | top 25% of $\log_{10}k$ | target-support extrapolation stress test; inputs and labels both shift, not a covariate-shift test |
| Morphology-family holdout (3 folds) | whole clusters of label-free image descriptors | input-defined shift: new rock textures, closest proxy for new formations available in this dataset |
| Index-block split | contiguous blocks of the file | proxy for shared generation history between training and test |

Then two remedies that need **no labels at deployment**:

- **Conformal out-of-distribution guard.** A novelty score (distance to the 10 nearest training volumes in the network's feature space, or in label-free image descriptors)
  is converted to a conformal p-value with the calibration set. Flagging $p\le\delta$ has a **guaranteed false-alarm rate $\le\delta$** on exchangeable data. Flagged rocks go to simulation.
- **Weighted conformal prediction** (Tibshirani et al., 2019) with a density ratio estimated from *unlabelled* deployment images. It restores coverage under covariate shift
  if the ratio is estimated well and the supports overlap. We test it under a controlled tilt of the porosity distribution, where the true ratio is known, and on the family holdout.

In [ ]:
# ============================== 13a. Coverage under the three shifts ==============================
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score
SHIFT_TAGS = [t for t in RUNS if RUNS[t]["group"] == "shift"]
EXP_LAB = {"shiftA": "label extrapolation", "block": "index blocks"}
EXP_LAB.update({f"family_f{i}": f"family fold {i}" for i in range(len(FAMILY_FOLDS))})
sh_rows = []
for tag in SHIFT_TAGS:
    Pm = load_preds(tag); cal, tr = Pm["cal"], Pm["tr"]
    parts = [("in-distribution test", Pm["te"])] + ([("shifted", Pm["ood"])] if "ood" in Pm else [])
    if tag == "block": parts = [("shifted", Pm["te"])]
    for part, te in parts:
        for j, pn in enumerate(PNAMES):
            y = te["y"][:, j]
            for m in ALL_M:
                lo, hi, _ = interval(m, cal, te, j, cfg.alpha, tr=tr)
                sh_rows.append(dict(exp=tag, label=EXP_LAB[tag], part=part, prop=pn, method=m, coverage=cov(y, lo, hi), width=float((hi - lo).mean()),
                                    sig_ratio=float(te["sig"][:, j].mean() / cal["sig"][:, j].mean()), rmse=reg_metrics(y, te["mu"][:, j])["RMSE"], n=len(y)))
SH = pd.DataFrame(sh_rows)
if len(SH):
    save_table(SH[SH.prop == "log_k"].pivot_table(index=["label", "part"], columns="method", values="coverage").reset_index(), "T24_shift_coverage_log_k")
    save_table(SH, "T24b_shift_coverage_all", show=False)
    print("Ratio of mean predicted sd (shifted / calibration), log k:")
    display(SH[(SH.prop == "log_k") & (SH.method == "split")][["label", "part", "sig_ratio", "rmse", "n"]].round(3))

In [ ]:
# ============================== 13b. Conformal out-of-distribution guard ==============================
def knn_score(fit_X, X, k=10):
    nn_ = NearestNeighbors(n_neighbors=k).fit(fit_X); return nn_.kneighbors(X)[0].mean(1)
def novelty(Pm, space):
    """Returns novelty scores for every part, fitted on training data only."""
    if space == "network features":
        f_tr = Pm["tr"]["feat"]; m_, s_ = f_tr.mean(0), f_tr.std(0) + 1e-6
        Z = {p: (Pm[p]["feat"] - m_) / s_ for p in Pm}
    else:
        Z = {p: DFEAT[Pm[p]["idx"]] for p in Pm}
    ref = Z["tr"] if len(Z["tr"]) <= 20000 else Z["tr"][np.random.default_rng(0).choice(len(Z["tr"]), 20000, replace=False)]
    return {p: knn_score(ref, Z[p]) for p in Pm if p != "tr"}
def conf_pvalue(s_cal, s): return (1 + (s_cal[None, :] >= s[:, None]).sum(1)) / (len(s_cal) + 1)
g_rows, GUARD = [], {}
for tag in [t for t in SHIFT_TAGS if "ood" in load_preds(t)]:
    Pm = load_preds(tag)
    for space in ("network features", "image descriptors"):
        sc = novelty(Pm, space); p_te, p_ood = conf_pvalue(sc["cal"], sc["te"]), conf_pvalue(sc["cal"], sc["ood"])
        GUARD[(tag, space)] = dict(sc=sc, p_te=p_te, p_ood=p_ood)
        auc = roc_auc_score(np.r_[np.zeros(len(sc["te"])), np.ones(len(sc["ood"]))], np.r_[sc["te"], sc["ood"]])
        fl_te, fl_ood = p_te <= cfg.guard_delta, p_ood <= cfg.guard_delta
        row = dict(exp=tag, label=EXP_LAB[tag], space=space, false_alarm_in_dist=fl_te.mean(), detection_shifted=fl_ood.mean(), auroc=auc)
        for m in ["split", "cqr", "mondrian"]:
            lo, hi, _ = interval(m, Pm["cal"], Pm["ood"], J_K, cfg.alpha, tr=Pm["tr"]); hit = (Pm["ood"]["y"][:, J_K] >= lo) & (Pm["ood"]["y"][:, J_K] <= hi)
            row[f"{m}_cov_all_shifted"] = hit.mean(); row[f"{m}_cov_unflagged_shifted"] = hit[~fl_ood].mean() if (~fl_ood).any() else np.nan
        g_rows.append(row)
GD = pd.DataFrame(g_rows)
if len(GD): save_table(GD, "T25_ood_guard")

In [ ]:
# ============================== 13c. Weighted conformal prediction ==============================
def wcq(s_cal, w_cal, w_te, alpha):
    """Weighted conformal threshold for each test point (Tibshirani et al. 2019): test mass sits at +inf."""
    o = np.argsort(s_cal); ss, cw = s_cal[o], np.cumsum(w_cal[o]); W = cw[-1]
    idx = np.searchsorted(cw, (1 - alpha) * (W + w_te) - 1e-12, side="left")
    return np.where(idx < len(ss), ss[np.minimum(idx, len(ss) - 1)], np.inf)
def est_ratio(X_cal, X_unl, X_eval, seed=0):
    """Density ratio p_target/p_cal by probabilistic classification on label-free descriptors."""
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(np.r_[X_cal, X_unl], np.r_[np.zeros(len(X_cal)), np.ones(len(X_unl))])
    f = lambda X: np.clip(clf.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)
    r = len(X_cal) / len(X_unl); g = lambda X: f(X) / (1 - f(X)) * r
    return g(X_cal), g(X_eval)
def ess(w): return w.sum()**2 / (w**2).sum()
def cqr_scores(d, j):
    a, b = q_pair(cfg.alpha); lo, hi = np.minimum(d["q"][:, j, a], d["q"][:, j, b]), np.maximum(d["q"][:, j, a], d["q"][:, j, b]); return lo, hi
_lp = np.log(PHI / (1 - PHI + 1e-9) + 1e-9); ZPOR = (_lp - _lp.mean()) / _lp.std()          # label-free tilt variable
BETAS = [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0]
w_rows = []
for s in range(min(3, cfg.n_seeds)):
    Pm = load_preds(f"main_s{s}"); cal, te = Pm["cal"], Pm["te"]; rng = np.random.default_rng(900 + s)
    half = rng.permutation(len(te["y"])); A, B = half[: len(half) // 2], half[len(half) // 2:]
    yc = cal["y"][:, J_K]; s_split = np.abs(yc - cal["mu"][:, J_K]); clo, chi = cqr_scores(cal, J_K); s_cqr = np.maximum(clo - yc, yc - chi)
    for beta in BETAS:
        for r in range(cfg.n_tilt_rep):
            def draw(pool, m):
                z = ZPOR[te["idx"][pool]]; p = np.exp(beta * (z - z.max())); return pool[rng.choice(len(pool), m, replace=True, p=p / p.sum())]
            ev = draw(B, min(1000, len(B))); un = draw(A, min(1000, len(A)))
            y = te["y"][ev, J_K]; mu = te["mu"][ev, J_K]; tlo, thi = cqr_scores(take(te, ev), J_K)
            w_true_c, w_true_e = np.exp(beta * ZPOR[cal["idx"]]), np.exp(beta * ZPOR[te["idx"][ev]])
            w_est_c, w_est_e = est_ratio(DFEAT[cal["idx"]], DFEAT[te["idx"][un]], DFEAT[te["idx"][ev]])
            out = {}
            q = cq(s_split, cfg.alpha); out[("split", "unweighted")] = (mu - q, mu + q)
            for wn, (wc, we) in (("true ratio", (w_true_c, w_true_e)), ("estimated ratio", (w_est_c, w_est_e))):
                qw = wcq(s_split, wc, we, cfg.alpha); out[("split", wn)] = (mu - qw, mu + qw)
                qw = wcq(s_cqr, wc, we, cfg.alpha); out[("cqr", wn)] = (tlo - qw, thi + qw)
            q = cq(s_cqr, cfg.alpha); out[("cqr", "unweighted")] = (tlo - q, thi + q)
            for (m, wn), (lo, hi) in out.items():
                fin = np.isfinite(hi - lo)
                w_rows.append(dict(seed=s, beta=beta, rep=r, method=m, weighting=wn, coverage=cov(y, lo, hi), width=float((hi - lo)[fin].mean()) if fin.any() else np.nan,
                                   frac_infinite=1 - fin.mean(), ess_cal=ess(w_est_c) if wn == "estimated ratio" else ess(w_true_c) if wn == "true ratio" else len(yc)))
WT = pd.DataFrame(w_rows)
save_table(WT.groupby(["method", "weighting", "beta"]).agg(coverage=("coverage", "mean"), coverage_sd=("coverage", "std"), width=("width", "mean"),
                                                           frac_infinite=("frac_infinite", "mean"), ess_cal=("ess_cal", "mean")).reset_index(), "T26_weighted_conformal_tilt", show=False)
wf_rows = []
for tag in [t for t in SHIFT_TAGS if t.startswith("family")]:
    Pm = load_preds(tag); cal, ood = Pm["cal"], Pm["ood"]; rng = np.random.default_rng(77)
    pr = rng.permutation(len(ood["y"])); A, B = pr[: len(pr) // 2], pr[len(pr) // 2:]
    yc = cal["y"][:, J_K]; s_split = np.abs(yc - cal["mu"][:, J_K])
    wc, we = est_ratio(DFEAT[cal["idx"]], DFEAT[ood["idx"][A]], DFEAT[ood["idx"][B]])
    y, mu = ood["y"][B, J_K], ood["mu"][B, J_K]
    q = cq(s_split, cfg.alpha); qw = wcq(s_split, wc, we, cfg.alpha); fin = np.isfinite(qw)
    hit_w = (y >= mu - qw) & (y <= mu + qw)
    wf_rows.append(dict(exp=EXP_LAB[tag], unweighted_coverage=cov(y, mu - q, mu + q), weighted_coverage=hit_w.mean(),
                        weighted_coverage_finite_only=hit_w[fin].mean() if fin.any() else np.nan,
                        weighted_frac_infinite=1 - fin.mean(), weighted_width_finite=float(2 * qw[fin].mean()) if fin.any() else np.nan, unweighted_width=2 * q,
                        effective_cal_size=ess(wc), n_cal=len(wc)))
WF = pd.DataFrame(wf_rows)
if len(WF): save_table(WF, "T26b_weighted_conformal_family_holdout")

In [ ]:
# ============================== Figures 13-15 ==============================
DIV = matplotlib.colors.LinearSegmentedColormap.from_list("div", ["#c2410c", "#eb6834", "#f3c6ad", "#ecebe7", "#b7d3f6", "#5598e7", "#1c5cab"])
def fig13():
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.0), gridspec_kw=dict(width_ratios=[1, 1, 1.25]))
    ax = axes[0]; d = SH[(SH.exp == "shiftA") & (SH.prop == "log_k")]
    ax.axvline(TARGET, color=INK, lw=0.8)
    for i, m in enumerate(ALL_M):
        a = d[(d.method == m) & (d.part == "in-distribution test")].coverage.values; b = d[(d.method == m) & (d.part == "shifted")].coverage.values
        if len(a) and len(b):
            ax.plot([a[0], b[0]], [i, i], color=GRID, lw=2.2, zorder=1)
            ax.plot(a[0], i, ls="none", marker=METHODS[m][2], ms=6, mfc=SURF, mec=MCOL[m], mew=1.3, zorder=2)
            ax.plot(b[0], i, ls="none", marker=METHODS[m][2], ms=6, color=MCOL[m], zorder=3)
    ax.set_yticks(range(len(ALL_M))); ax.set_yticklabels([MLAB[m] for m in ALL_M]); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set_xlabel("coverage of log-permeability"); ax.set_title("Label extrapolation\nopen: in-distribution; filled: withheld top 25%"); ax.set_xlim(-0.02, 1.02)
    ax = axes[1]; fams = [t for t in SHIFT_TAGS if t.startswith("family")] + (["block"] if "block" in SHIFT_TAGS else [])
    ax.axvline(TARGET, color=INK, lw=0.8)
    for i, m in enumerate(ALL_M):
        for k_, t in enumerate(fams):
            v = SH[(SH.exp == t) & (SH.prop == "log_k") & (SH.method == m) & (SH.part == "shifted")].coverage.values
            if len(v): ax.plot(v[0], i + (k_ - (len(fams) - 1) / 2) * 0.17, ls="none", marker=["o", "s", "^", "D"][k_ % 4], ms=5, color=MCOL[m],
                               mfc=SURF if t == "block" else MCOL[m])
    ax.set_yticks(range(len(ALL_M))); ax.set_yticklabels([]); ax.invert_yaxis(); ax.grid(axis="y", visible=False); ax.set_xlim(-0.02, 1.02)
    ax.set_xlabel("coverage of log-permeability"); ax.set_title("Withheld families and index blocks")
    hs = [matplotlib.lines.Line2D([], [], ls="none", marker=["o", "s", "^", "D"][k_ % 4], color=INK2, mfc=SURF if t == "block" else INK2, label=EXP_LAB[t]) for k_, t in enumerate(fams)]
    ax.legend(handles=hs, fontsize=7, loc="lower left")
    ax = axes[2]; ft = [t for t in SHIFT_TAGS if t.startswith("family")]
    M = SH[SH.exp.isin(ft) & (SH.part == "shifted")].groupby(["method", "prop"]).coverage.mean().unstack().reindex(index=ALL_M, columns=PNAMES)
    vmin = min(TARGET - 0.05, float(np.nanmin(M.values)))
    im = ax.imshow(M.values, cmap=DIV, norm=matplotlib.colors.TwoSlopeNorm(vcenter=TARGET, vmin=vmin, vmax=1.0), aspect="auto")
    for (a, b), v in np.ndenumerate(M.values):
        dark = v < TARGET - 0.6 * (TARGET - vmin)
        ax.text(b, a, f"{v:.2f}", ha="center", va="center", fontsize=7.5, color="#ffffff" if dark else INK)
    prop_ticks(ax); ax.set_yticks(range(len(ALL_M))); ax.set_yticklabels([MLAB[m] for m in ALL_M], fontsize=7.5)
    ax.grid(False); ax.set_title("Coverage on withheld families (mean of folds)")
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02); cb.set_label("coverage (grey = target)"); cb.outline.set_visible(False)
    cb.set_ticks(sorted({round(vmin, 2), round((vmin + TARGET) / 2, 2), TARGET, 1.0}))
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F13_distribution_shift")
if len(SH): fig13()

def fig14():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.7))
    tag0 = next(t for t in SHIFT_TAGS if t.startswith("family")); G0 = GUARD[(tag0, "network features")]
    ax = axes[0]; allv = np.r_[G0["sc"]["te"], G0["sc"]["ood"]]; bins = np.linspace(np.percentile(allv, 0.5), np.percentile(allv, 99.5), 50)
    ax.hist(G0["sc"]["te"], bins=bins, density=True, color="#b7d3f6", label="in-distribution test")
    ax.hist(G0["sc"]["ood"], bins=bins, density=True, histtype="step", lw=1.6, color="#eb6834", label=f"withheld ({EXP_LAB[tag0]})")
    thr = np.quantile(G0["sc"]["cal"], 1 - cfg.guard_delta); ax.axvline(thr, color=INK, lw=1, ls=(0, (4, 3)))
    ax.text(thr, ax.get_ylim()[1] * 1.02, f"flag if p ≤ {cfg.guard_delta}", fontsize=7.5, va="bottom", ha="center")
    ax.set_xlabel("novelty: mean distance to 10 nearest training volumes"); ax.set_ylabel("density"); ax.set_title("Novelty in network feature space", pad=14)
    ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    ax = axes[1]; exps = GD.label.unique(); x = np.arange(len(exps))
    for k_, (space, mk) in enumerate((("network features", "o"), ("image descriptors", "s"))):
        d = GD[GD.space == space].set_index("label").reindex(exps)
        ax.plot(x + (k_ - 0.5) * 0.2, d.detection_shifted, ls="none", marker=mk, ms=7, color="#2a78d6", label=f"detected shifted, {space}")
        ax.plot(x + (k_ - 0.5) * 0.2, d.false_alarm_in_dist, ls="none", marker=mk, ms=7, mfc=SURF, color="#eb6834", label=f"false alarm, {space}")
    ax.axhline(cfg.guard_delta, color=INK, lw=0.8, ls=(0, (4, 3))); ax.text(len(exps) - 0.5, cfg.guard_delta, "guaranteed\nfalse-alarm bound", fontsize=7, va="bottom", ha="right")
    ax.set_xticks(x); ax.set_xticklabels([e.replace(" ", "\n") for e in exps], fontsize=7.5); ax.set_ylabel("fraction flagged"); ax.set_ylim(-0.02, 1.02)
    ax.set_title("Label-free guard", pad=14); ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2)
    ax = axes[2]; d = GD[GD.space == "network features"].set_index("label").reindex(exps); ax.axhline(TARGET, color=INK, lw=0.8)
    for k_, m in enumerate(["split", "cqr", "mondrian"]):
        xx = x + (k_ - 1) * 0.22
        ax.vlines(xx, d[f"{m}_cov_all_shifted"], d[f"{m}_cov_unflagged_shifted"], color=MCOL[m], lw=1.4, alpha=0.6)
        ax.plot(xx, d[f"{m}_cov_all_shifted"], ls="none", marker=METHODS[m][2], ms=6, mfc=SURF, mec=MCOL[m], mew=1.3)
        ax.plot(xx, d[f"{m}_cov_unflagged_shifted"], ls="none", marker=METHODS[m][2], ms=6, color=MCOL[m], label=MLAB[m])
    ax.set_xticks(x); ax.set_xticklabels([e.replace(" ", "\n") for e in exps], fontsize=7.5); ax.set_ylabel("coverage on shifted volumes (log k)")
    ax.set_title("Coverage: all (open) vs not flagged (filled)", pad=14); ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3); ax.set_ylim(-0.02, 1.02)
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F14_ood_guard")
if len(GD): fig14()

def fig15():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.7))
    wcol = {"unweighted": "#2a78d6", "true ratio": "#eb6834", "estimated ratio": "#1baf7a"}
    ax = axes[0]; Pm = load_preds("main_s0"); b0 = min(BETAS); p = np.exp(b0 * ZPOR[Pm["te"]["idx"]]); p /= p.sum()
    bins = np.linspace(0, PHI.max(), 40)
    ax.hist(PHI[Pm["cal"]["idx"]], bins=bins, density=True, color="#b7d3f6", label="calibration")
    ax.hist(PHI[Pm["te"]["idx"]], bins=bins, weights=p * len(p), density=True, histtype="step", lw=1.6, color="#eb6834", label=f"deployment, tilt β = {b0}")
    ax.set_xlabel("porosity (label-free)"); ax.set_ylabel("density"); ax.set_title("Controlled covariate shift"); ax.legend(fontsize=7.5, loc="upper right")
    for ax, col, ttl in ((axes[1], "coverage", "Coverage under the tilt"), (axes[2], "width", "Width under the tilt (finite intervals)")):
        if col == "coverage": ax.axhline(TARGET, color=INK, lw=0.8)
        for m, ls, mk in (("split", "-", "o"), ("cqr", (0, (4, 2)), "^")):
            for wn in ["unweighted", "true ratio", "estimated ratio"]:
                d = WT[(WT.method == m) & (WT.weighting == wn)].groupby("beta")[col]
                ax.errorbar(d.mean().index + {"unweighted": -0.04, "true ratio": 0, "estimated ratio": 0.04}[wn], d.mean().values,
                            yerr=d.std().values if col == "coverage" else None, color=wcol[wn], ls=ls, marker=mk, ms=4.5, capsize=1.5,
                            mfc=wcol[wn] if m == "split" else SURF, label=f"{'Split' if m == 'split' else 'CQR'}, {wn}")
        ax.set_xlabel("tilt β (negative = toward tighter rocks)"); ax.set_title(ttl)
        ax.set_ylabel("coverage of log-permeability" if col == "coverage" else "mean width, log-permeability")
    h_, l_ = axes[1].get_legend_handles_labels()
    fig.legend(h_, l_, fontsize=7.5, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.02))
    panel_letters(axes); fig.tight_layout(rect=(0, 0.1, 1, 1)); savefig(fig, "F15_weighted_conformal")
fig15()

## 14. New contribution: risk-controlled simulation triage

The practical question is not "how wide is the interval" but "**which network predictions can I accept, and which rocks must I still simulate?**"

**Rule.** Accept the network's permeability for a rock if an uncertainty score $u(x)$ is at most a threshold $t$; otherwise run the pore-scale simulation.
An accepted prediction is a *bad accept* if its error exceeds a tolerance, $|\hat\mu(x)-y|>\log_{10}f$ (a factor $f$ on permeability).

**Guarantee.** We choose $t$ on the calibration set with Learn-then-Test (Angelopoulos et al., 2021): thresholds from a grid fixed in advance
(quantiles of the score on the monitoring slice, which is disjoint from calibration and test) are tested in increasing order with the exact binomial p-value
for $H_t:\ P(u(x)\le t,\ \text{bad})>\varepsilon$, stopping at the first that is not rejected at level $\delta$. With probability at least $1-\delta$ over the calibration draw,

$$P\big(\text{accepted and error} > \log_{10} f\big)\ \le\ \varepsilon .$$

The same rule with conformal risk control (an expectation-level guarantee) is reported alongside. The output is the **fraction of simulations avoided** at a guaranteed error rate.

**Scores compared.** Gaussian-head sd, CQR band width, MC-dropout total sd, feature-space novelty, and a random score (accepting at random). The oracle uses the true error.
Under distribution shift the guarantee no longer holds; we measure what happens with and without the out-of-distribution guard of Section 13.

In [ ]:
# ============================== 14a. Triage library ==============================
TRI_COL = {"gauss_sd": ("Gaussian-head sd", "#2a78d6", "o"), "cqr_w": ("CQR band width", "#eb6834", "s"), "mc_sd": ("MC-dropout total sd", "#1baf7a", "^"),
           "novelty": ("Feature novelty", "#6250d6", "D"), "random": ("Random (no information)", "#8a8984", "x")}
def tri_scores(Pm, part, j=J_K, nov=None, seed=0):
    d = Pm[part]; a, b = q_pair(cfg.alpha)
    out = dict(gauss_sd=d["sig"][:, j], cqr_w=np.abs(d["q"][:, j, b] - d["q"][:, j, a]),
               mc_sd=np.sqrt(d["mc_vm"][:, j] + d["mc_mv"][:, j]) if "mc_vm" in d else d["sig"][:, j],
               random=np.random.default_rng(1000 * seed + {"cal": 1, "te": 2, "ood": 3}.get(part, 4)).random(len(d["y"])))
    if nov is not None: out["novelty"] = nov[part]
    return out
def tri_grid(u_ref, n_grid=200):
    """Threshold grid fixed before calibration: quantiles of the score on data disjoint from calibration and test."""
    return np.unique(np.quantile(u_ref, np.linspace(0, 1, n_grid + 1)[1:]))
def ltt_threshold(u, bad, eps, delta, grid):
    """Largest t in the fixed ascending grid such that all t' <= t are rejected by exact binomial tests: P(u<=t, bad) <= eps w.p. 1-delta."""
    n = len(u)
    o = np.argsort(u); us, bs = u[o], bad[o].astype(int); cb = np.cumsum(bs)
    k = np.where(np.searchsorted(us, grid, side="right") > 0, cb[np.maximum(np.searchsorted(us, grid, side="right") - 1, 0)], 0)
    pv = st.binom.cdf(k, n, eps); ok = pv <= delta
    if not ok[0]: return -np.inf
    first_fail = np.argmin(ok) if (~ok).any() else len(ok)
    return grid[first_fail - 1]
def crc_threshold(u, bad, eps):
    n = len(u); o = np.argsort(u); us, cb = u[o], np.cumsum(bad[o].astype(int))
    ok = (cb + 1) / (n + 1) <= eps
    return us[np.where(ok)[0].max()] if ok.any() else -np.inf
def tri_eval(u, bad, t): acc = u <= t; return dict(accepted=acc.mean(), bad_accept=(acc & bad).mean(), err_among_accepted=(bad[acc].mean() if acc.any() else 0.0))
# Headline operating point. A loose tolerance is uninformative when the network is accurate (almost every prediction is
# within it, so every rule accepts everything); 1.25 (25% on k) is where the scores separate. All tolerances are in T27.
F0, E0 = 1.25, 0.05

In [ ]:
# ============================== 14b. In-distribution triage over seeds ==============================
tri_rows, NOV_MAIN = [], {}
for s, tag in enumerate(tqdm(MAIN, desc="triage")):
    Pm = load_preds(tag); NOV_MAIN[tag] = nov = novelty(Pm, "network features")
    U_cal, U_te, U_val = tri_scores(Pm, "cal", nov=nov, seed=s), tri_scores(Pm, "te", nov=nov, seed=s), tri_scores(Pm, "val", nov=nov, seed=s)
    e_cal = np.abs(Pm["cal"]["mu"][:, J_K] - Pm["cal"]["y"][:, J_K]); e_te = np.abs(Pm["te"]["mu"][:, J_K] - Pm["te"]["y"][:, J_K])
    for f in cfg.tol_factors:
        tol = np.log10(f); bc, bt = e_cal > tol, e_te > tol
        for eps in cfg.risk_levels:
            for sc in TRI_COL:
                for rule, fn in (("LTT", lambda u, b: ltt_threshold(u, b, eps, cfg.ltt_delta, tri_grid(U_val[sc]))), ("CRC", lambda u, b: crc_threshold(u, b, eps))):
                    t = fn(U_cal[sc], bc); r = tri_eval(U_te[sc], bt, t)
                    tri_rows.append(dict(seed=s, tol_factor=f, eps=eps, score=sc, rule=rule, base_error_rate=bt.mean(), **r))
TRI = pd.DataFrame(tri_rows)
T27 = TRI[TRI.rule == "LTT"].groupby(["tol_factor", "eps", "score"], sort=False).agg(base_error_rate=("base_error_rate", "mean"),
        simulations_avoided=("accepted", "mean"), simulations_avoided_sd=("accepted", "std"), bad_accept_rate=("bad_accept", "mean"),
        bad_accept_max_over_seeds=("bad_accept", "max"), error_among_accepted=("err_among_accepted", "mean")).reset_index()
T27["score"] = T27.score.map(lambda k: TRI_COL[k][0]); save_table(T27, "T27_triage_in_distribution", show=False)
display(T27[T27.tol_factor == F0].round(3))
print("Base error rate (share of test predictions outside the tolerance) by tolerance factor:",
      TRI.groupby("tol_factor").base_error_rate.mean().round(4).to_dict())
save_table(TRI[TRI.rule == "CRC"].groupby(["tol_factor", "eps", "score"], sort=False)[["accepted", "bad_accept"]].mean().reset_index(), "T27b_triage_crc", show=False)

# ---- guarantee check. Calibrate on random half-size subsamples of the calibration set, evaluate on the independent test set
# of the same seed (never on the complement of a fixed pool, which couples calibration and test rates and biases the check).
# Under the guarantee, P(true bad-accept rate > eps) <= delta; the test rate adds binomial noise, so we also count draws
# that exceed eps by more than 1.645 binomial standard errors (expected at most about delta + 5% if the guarantee holds).
rr = []
for s, tag in enumerate(MAIN):
    Pm = load_preds(tag); nov = NOV_MAIN[tag]; rng = np.random.default_rng(31 + s)
    Uc, Ut, Uv = (tri_scores(Pm, p, nov=nov, seed=s) for p in ("cal", "te", "val"))
    bc = np.abs(Pm["cal"]["mu"][:, J_K] - Pm["cal"]["y"][:, J_K]) > np.log10(F0); bt = np.abs(Pm["te"]["mu"][:, J_K] - Pm["te"]["y"][:, J_K]) > np.log10(F0)
    n_sub = len(bc) // 2
    for r in range(max(20, cfg.n_repart // len(MAIN))):
        c_ = rng.choice(len(bc), n_sub, replace=False)
        for k in ["gauss_sd", "cqr_w", "mc_sd", "novelty"]:
            t = ltt_threshold(Uc[k][c_], bc[c_], E0, cfg.ltt_delta, tri_grid(Uv[k])); rr.append(dict(seed=s, rep=r, score=k, n_cal=n_sub, n_test=len(bt), **tri_eval(Ut[k], bt, t)))
TRR = pd.DataFrame(rr); SE0 = np.sqrt(E0 * (1 - E0) / TRR.n_test.iloc[0])
save_table(TRR.groupby("score").agg(mean_bad_accept=("bad_accept", "mean"), frac_draws_above_eps=("bad_accept", lambda x: (x > E0).mean()),
                                    frac_draws_above_eps_plus_1645se=("bad_accept", lambda x: (x > E0 + 1.645 * SE0).mean()),
                                    mean_accepted=("accepted", "mean"), n_draws=("rep", "size")).reset_index().assign(eps=E0, delta=cfg.ltt_delta, n_cal=TRR.n_cal.iloc[0]),
           "T28_triage_guarantee_check")
U_pool = {k: None for k in ["gauss_sd", "cqr_w", "mc_sd", "novelty"]}

In [ ]:
# ============================== 14c. Triage under shift, with and without the guard ==============================
ts_rows = []
for tag in [t for t in SHIFT_TAGS if "ood" in load_preds(t)]:
    Pm = load_preds(tag); nov = novelty(Pm, "network features"); pv = conf_pvalue(nov["cal"], nov["ood"])
    U_cal, U_ood, U_val = tri_scores(Pm, "cal", nov=nov), tri_scores(Pm, "ood", nov=nov), tri_scores(Pm, "val", nov=nov)
    bc = np.abs(Pm["cal"]["mu"][:, J_K] - Pm["cal"]["y"][:, J_K]) > np.log10(F0); bo = np.abs(Pm["ood"]["mu"][:, J_K] - Pm["ood"]["y"][:, J_K]) > np.log10(F0)
    for eps in cfg.risk_levels:
        for sc in ["gauss_sd", "cqr_w", "mc_sd", "novelty"]:
            t = ltt_threshold(U_cal[sc], bc, eps, cfg.ltt_delta, tri_grid(U_val[sc]))
            for guard in (False, True):
                am = (U_ood[sc] <= t) & (~(pv <= cfg.guard_delta) if guard else True)
                ts_rows.append(dict(exp=EXP_LAB[tag], eps=eps, score=TRI_COL[sc][0], guard="with guard" if guard else "no guard",
                                    accepted=am.mean(), bad_accept=(am & bo).mean(), base_error_rate=bo.mean()))
TS = pd.DataFrame(ts_rows)
if len(TS): save_table(TS, "T29_triage_under_shift", show=False); display(TS[(TS.eps == E0)].round(3))

In [ ]:
# ============================== Figure 16: triage ==============================
def fig16():
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.7), gridspec_kw=dict(width_ratios=[1, 1, 1, 1.15]))
    ax = axes[0]; d = TRI[(TRI.rule == "LTT") & (TRI.tol_factor == F0)]
    for sc, (lab, col, mk) in TRI_COL.items():
        g = d[d.score == sc].groupby("eps").accepted
        ax.errorbar(np.array(g.mean().index) * 100, g.mean() * 100, yerr=g.std() * 100, color=col, marker=mk, ms=5.5, capsize=2, label=lab, ls="-" if sc != "random" else (0, (2, 2)))
    ax.set_xlabel("guaranteed bad-accept rate ε (%)"); ax.set_ylabel("simulations avoided (%)"); ax.set_title(f"Simulations avoided\n(error tolerance: factor {F0:g} on k)")
    ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    ax = axes[1]; Pm = load_preds("main_s0"); U = tri_scores(Pm, "te", nov=NOV_MAIN["main_s0"]); e = np.abs(Pm["te"]["mu"][:, J_K] - Pm["te"]["y"][:, J_K]); bad = e > np.log10(F0)
    fr = np.linspace(0.02, 1, 50)
    for sc, (lab, col, mk) in list(TRI_COL.items()) + [("oracle", ("Oracle (true error)", INK, ""))]:
        u = e if sc == "oracle" else U[sc]; o = np.argsort(u); cb = np.cumsum(bad[o]) / np.arange(1, len(o) + 1)
        ax.plot(fr * 100, cb[(fr * len(o)).astype(int) - 1] * 100, color=col, lw=1.6, ls=(0, (2, 2)) if sc in ("random", "oracle") else "-", label=lab if sc == "oracle" else None)
    ax.set_xlabel("fraction of rocks accepted (%)"); ax.set_ylabel(f"error > factor {F0:g} among accepted (%)"); ax.set_title("Selective risk (test, seed 0)"); ax.legend(fontsize=7, loc="upper left")
    ax = axes[2]
    for sc in U_pool:
        lab, col, mk = TRI_COL[sc]; v = TRR[TRR.score == sc].bad_accept * 100
        ax.hist(v, bins=25, histtype="step", lw=1.5, color=col, label=f"{lab}: {(v > (E0 + 1.645 * SE0) * 100).mean() * 100:.1f}% above ε + 1.645 se")
    ax.axvline(E0 * 100, color=INK, lw=1); ax.text(E0 * 100, ax.get_ylim()[1] * 0.97, " ε", va="top", fontsize=8)
    ax.axvline((E0 + 1.645 * SE0) * 100, color=INK, lw=0.8, ls=(0, (4, 3)))
    ax.set_xlabel("bad-accept rate on independent test set (%)"); ax.set_ylabel(f"count over {len(TRR) // 4} calibration draws"); ax.set_title(f"Guarantee check (δ = {cfg.ltt_delta})")
    ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=1)
    ax = axes[3]
    if len(TS):
        d = TS[(TS.eps == E0) & (TS.score == "Gaussian-head sd")]; exps = d.exp.unique(); x = np.arange(len(exps))
        for k_, (gname, mfc) in enumerate((("no guard", SURF), ("with guard", "#2a78d6"))):
            dd = d[d.guard == gname].set_index("exp").reindex(exps)
            ax.plot(x + (k_ - 0.5) * 0.2, dd.bad_accept * 100, ls="none", marker="o", ms=7, mfc=mfc, mec="#2a78d6", mew=1.3, label=f"bad accepts, {gname}")
            ax.plot(x + (k_ - 0.5) * 0.2, dd.accepted * 100, ls="none", marker="s", ms=6, mfc=mfc if mfc == SURF else "#1baf7a", mec="#1baf7a", mew=1.3, label=f"accepted, {gname}")
        ax.axhline(E0 * 100, color=INK, lw=0.9, ls=(0, (4, 3))); ax.text(-0.4, E0 * 100, " ε", va="bottom", fontsize=8); ax.set_ylim(bottom=0)
        ax.set_xticks(x); ax.set_xticklabels([e.replace(" ", "\n") for e in exps], fontsize=7.5); ax.set_ylabel("% of shifted rocks")
        ax.set_title(f"Under shift (ε = {E0:.0%}, Gaussian-head sd)"); ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2)
    panel_letters(axes); fig.tight_layout(); savefig(fig, "F16_risk_controlled_triage")
fig16()

## 15. Cost, requirements, reviewer map, key numbers and release pack (Reviewer 1; Reviewer 2, comments 12 and 17)

In [ ]:
# ============================== 15a. Measured cost and what each method needs ==============================
logs = pd.DataFrame([dict(run=t, **{k: v for k, v in run_log(t).items() if k in ("mode", "variant", "group", "train_minutes", "ms_per_volume_deterministic",
                                                                                   "ms_per_volume_per_mc_pass", "device", "n_params")}) for t in RUNS])
Pm = load_preds("main_s0"); n_cal = len(Pm["cal"]["y"]); t0 = time.perf_counter()
for _ in range(200):
    for m in ("split", "norm", "cqr", "mondrian"): interval(m, Pm["cal"], Pm["te"], J_K, cfg.alpha, tr=Pm["tr"])
cal_ms = (time.perf_counter() - t0) / 200 / 4 * 1e3
main_logs = logs[logs.group == "main"]
cost = pd.DataFrame([
    dict(item=f"preprocessing of {N:,} volumes ({FULL_RES}^3 -> 2x{cfg.work_res}^3), once", value=META["preprocess_s"] / 60, unit="min"),
    dict(item="training, one jointly trained network", value=main_logs.train_minutes.mean(), unit="min"),
    dict(item="inference, deterministic (all heads)", value=main_logs.ms_per_volume_deterministic.mean(), unit="ms / volume"),
    dict(item=f"inference, MC-dropout ({cfg.mc_T} passes)", value=main_logs.ms_per_volume_per_mc_pass.mean() * cfg.mc_T, unit="ms / volume"),
    dict(item=f"conformal calibration + intervals for {len(Pm['te']['y']):,} test volumes (n_cal = {n_cal:,})", value=cal_ms, unit="ms, total"),
    dict(item=f"deep ensemble ({cfg.ens_M} networks), training", value=logs[logs.run.str.startswith("gauss_s0_m")].train_minutes.sum(), unit="min"),
    dict(item="all runs in this notebook, training", value=logs.train_minutes.sum() / 60, unit="h"),
])
cost["hardware"] = f"{GPU_NAME}; {os.cpu_count()} CPU cores; torch {torch.__version__}; mixed precision {'on' if AMP else 'off'}"
save_table(cost, "T30_measured_cost", floatfmt=3)
req = pd.DataFrame([
    dict(method="Split conformal", needs="point prediction only", retraining_needed="no", guarantee="marginal, per property"),
    dict(method="Normalized conformal", needs="point prediction + scale estimate (here the Gaussian head; MC-dropout or ensemble spread also work)", retraining_needed="yes, if the model has no scale output", guarantee="marginal, per property"),
    dict(method="CQR", needs="lower and upper quantile predictions", retraining_needed="yes, if the model has no quantile outputs", guarantee="marginal, per property"),
    dict(method="Mondrian conformal", needs="point prediction + a grouping fixed before calibration", retraining_needed="no", guarantee="within each group"),
    dict(method="Max-score joint conformal", needs="point predictions (+ scales for the adaptive version)", retraining_needed="no (fixed scale)", guarantee="joint over properties"),
    dict(method="Conformal OOD guard", needs="a novelty score fitted on training data", retraining_needed="no", guarantee="false-alarm rate <= delta (no labels needed)"),
    dict(method="Risk-controlled triage", needs="any uncertainty score + calibration labels", retraining_needed="no", guarantee="P(accepted and error > tolerance) <= eps, w.p. >= 1 - delta"),
    dict(method="Gaussian head / MC-dropout / ensemble", needs="variance output / dropout / several networks", retraining_needed="depends", guarantee="none"),
])
save_table(req, "T31_method_requirements")

In [ ]:
# ============================== 15b. Where each reviewer comment is answered ==============================
REVMAP = [
    ("R1", "Novelty: known methods, one dataset, two properties", "Six simulated properties; joint guarantee (S12); label-free OOD guard (S13b); weighted conformal (S13c); risk-controlled simulation triage (S14)", "T23, T25-T29; F12, F14-F16"),
    ("R1", "Labels are simulations; OOD test only withholds a label tail", "Claims restated as coverage of DeePore reference labels; input-defined family holdout and index-block split added; labelled-free guard", "T24-T26; F13-F15"),
    ("R1", "Repository not reproducible", "Data rebuilt from the public Zenodo file with checksum; split indices, checkpoints, predictions, tables, figures and environment exported", "S4, S15c; release zip"),
    ("R2-1", "4,000 at 64^3 vs 17,700 at 256^3; preprocessing; pairing", "All valid volumes; exclusion counts; no crop; porosity-conserving two-channel coarse-graining; connectivity check; input ablation", "T01, T02, T12, T13; F01, F03"),
    ("R2-2", "Related samples across splits", "Index-structure check, index-block split, morphology-family holdout; exchangeability within the collection separated from new rocks", "T03, T24; F02, F13"),
    ("R2-3", "Mondrian partition from the calibration set", "Group edges from training predictions (exact group-conditional guarantee); comparison with the original rule; score and binning stated", "S7, T18; F10"),
    ("R2-4", "Joint vs marginal coverage", "All guarantees stated per property; joint coverage measured; Bonferroni and max-score joint conformal", "T23; F12"),
    ("R2-5", "Figures, tables and sources disagree", "Every figure and table is generated from the same saved predictions in one run", "all T*, F*"),
    ("R2-6", "Calibration size up to 1,600 leaves no test data", "Sizes drawn from the untouched cal+test pool of a fixed predictor; counts per setting", "T14, T15; F09"),
    ("R2-7", "Coverage error averaged before taking absolute value", "Absolute error per property and seed, then averaged; maximum also reported", "T07"),
    ("R2-8", "One seed, one split; worst-slice noise; validity wording", f"{cfg.n_seeds} independent training seeds and splits; paired differences; per-bin tables; exact-law test of the implementation", "T06-T09, T15; F06, F08, F09"),
    ("R2-9", "MC-dropout variance, passes, BN/dropout modes, auxiliary heads, Gaussian over-coverage", "Means-only and total-variance MC-dropout to 100 passes; modes stated; mean-only and dedicated-Gaussian ablations; ensemble", "T20-T22; F11"),
    ("R2-10", "OOD experiment details, Mondrian missing, framing", "Cut-off, counts, retraining stated; all methods and properties; called a target-support stress test; label-free guard instead of label-based gap", "T04, T24, T25; F13, F14"),
    ("R2-11", "MAE/RMSE/R2; porosity baseline; units; physical widths", "Metrics per property with descriptor and Kozeny-Carman baselines; porosity shown to be an exact count and dropped as a target; px^2 units and conversion; fold-factor intervals", "T05, T11, T12; F04, F07"),
    ("R2-12", "'Any predictor without retraining' too broad; cost", "Requirements per method; measured cost on the stated hardware", "T30, T31"),
    ("R2-13", "Equations, quantile convention, Algorithm 1", "Order-statistic definition, ties, infinite threshold, small-group fallback stated", "S7"),
    ("R2-14", "Fixed 0.05/0.95 heads across the sweep; crossing; negative adjustments", "Ten quantile heads with matched pair per level; fixed-pair comparison; crossing and inversion counts", "T10, T19; F10"),
    ("R2-15", "Figure omissions and errors", "All four conformal methods in every figure; width distributions; no clipped legends", "F05-F09"),
    ("R2-16", "Terminology, duplicated text, citations", "Manuscript text (not in this notebook)", "-"),
    ("R2-17", "Immutable release", "Release zip with manifest and hashes; tag the repository at the commit that produced it", "S15c"),
]
save_table(pd.DataFrame(REVMAP, columns=["reviewer_comment", "issue", "what_this_notebook_does", "evidence"]), "T32_reviewer_comment_map")

In [ ]:
# ============================== 15c. Key numbers for the manuscript ==============================
def pick(df, **kw):
    d = df
    for k, v in kw.items(): d = d[d[k] == v]
    return d
KEY = {}
KEY["data"] = dict(n_volumes=int(N), source=META["source"], full_res=FULL_RES, work_res=cfg.work_res, n_seeds=cfg.n_seeds, n_cal=N_CAL, n_test=N_TEST)
KEY["accuracy_R2"] = {pn: round(float(ACC[(ACC.prop == pn) & ACC.model.str.startswith("3D CNN")].R2.mean()), 4) for pn in PNAMES}
KEY["mean_abs_cov_error_pp"] = {r.method: round(float(r.mean_abs_cov_error_pp), 2) for r in T07.itertuples()}
KEY["worst_porosity_group_cov_log_k"] = {m: round(float(pick(T06, prop="log_k", method=m).worst_group_porosity.iloc[0]), 3) for m in CONF}
KEY["width_rel_split"] = {r.method: round(float(r.width_relative_to_split), 3) for r in T07.itertuples()}
KEY["joint"] = {f"{r.set} | {r.method}": round(float(r.joint_coverage), 3) for r in T23.itertuples()}
if len(GD): KEY["guard"] = {f"{r.label} | {r.space}": dict(false_alarm=round(float(r.false_alarm_in_dist), 3), detection=round(float(r.detection_shifted), 3), auroc=round(float(r.auroc), 3)) for r in GD.itertuples()}
KEY[f"triage_tol{F0:g}_eps{E0:g}"] = {r.score: dict(avoided=round(float(r.simulations_avoided), 3), bad_accept=round(float(r.bad_accept_rate), 4))
                           for r in T27[(T27.tol_factor == F0) & (T27.eps == E0)].itertuples()}
KEY["triage_eps1pct"] = {f"factor {f:g} | {r.score}": round(float(r.simulations_avoided), 3)
                         for f in cfg.tol_factors for r in T27[(T27.tol_factor == f) & (T27.eps == 0.01)].itertuples()}
save_json(KEY, os.path.join(D["tables"], "key_numbers.json"))
print(json.dumps(KEY, indent=1, default=str)[:6000])

best = T27[(T27.tol_factor == F0) & (T27.eps == E0) & (T27.score != "Random (no information)")].sort_values("simulations_avoided").iloc[-1]
rnd = T27[(T27.tol_factor == F0) & (T27.eps == E0) & (T27.score == "Random (no information)")].iloc[0]
cm = T07.set_index("method")
display(Markdown(f"""
**Draft results paragraph (generated from the tables; check every number before use).**
Across {cfg.n_seeds} independent training seeds and splits of {N:,} DeePore volumes and six pore-network properties, the four conformal predictors
had mean absolute property-wise coverage errors of {cm.loc['split','mean_abs_cov_error_pp']:.1f} (split), {cm.loc['norm','mean_abs_cov_error_pp']:.1f} (normalized),
{cm.loc['cqr','mean_abs_cov_error_pp']:.1f} (CQR) and {cm.loc['mondrian','mean_abs_cov_error_pp']:.1f} (Mondrian) percentage points at the {TARGET:.0%} target,
against {cm.loc['gauss','mean_abs_cov_error_pp']:.1f} for the Gaussian head and {cm.loc['mcd','mean_abs_cov_error_pp']:.1f} / {cm.loc['mcd_tot','mean_abs_cov_error_pp']:.1f}
for MC-dropout (means / total variance). For a tolerance of a factor {F0:g} on permeability and a guaranteed bad-accept rate of {E0:.0%} (δ = {cfg.ltt_delta}),
risk-controlled triage with the {best.score.lower()} accepted {best.simulations_avoided:.0%} of predictions, against {rnd.simulations_avoided:.0%} for an uninformed rule.
"""))

In [ ]:
# ============================== 15d. Release pack ==============================
env = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
with open(os.path.join(D["logs"], "requirements_frozen.txt"), "w") as f: f.write(env)
save_json(dict(python=sys.version, platform=platform.platform(), torch=torch.__version__, cuda=torch.version.cuda, gpu=GPU_NAME, numpy=np.__version__,
               scipy=scipy.__version__, sklearn=sklearn.__version__, pandas=pd.__version__, h5py=h5py.__version__, profile=PROFILE, data=META, timers=TIMERS),
          os.path.join(D["logs"], "environment.json"))
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""): h.update(ch)
    return h.hexdigest()
REL = os.path.join(ROOT, f"release_{PROFILE}.zip"); man = []
with zipfile.ZipFile(REL, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ["tables", "figures", "splits", "logs", "preds"]:
        for p in sorted(glob.glob(os.path.join(D[sub], "*"))):
            z.write(p, os.path.join(sub, os.path.basename(p))); man.append(dict(file=f"{sub}/{os.path.basename(p)}", bytes=os.path.getsize(p), sha256=sha256(p)))
    for p in [os.path.join(D["proc"], "exclusions.csv"), os.path.join(D["proc"], "connectivity.csv"), os.path.join(D["proc"], "proc_meta.json")]:
        z.write(p, "data/" + os.path.basename(p)); man.append(dict(file="data/" + os.path.basename(p), bytes=os.path.getsize(p), sha256=sha256(p)))
    z.writestr("data/selected_h5_indices.csv", "h5_index\n" + "\n".join(map(str, sel.tolist())))
    ck = [dict(file=f"ckpt/{os.path.basename(p)}", bytes=os.path.getsize(p), sha256=sha256(p)) for p in sorted(glob.glob(os.path.join(D["ckpt"], "*.pt")))]
    z.writestr("MANIFEST.json", json.dumps(dict(files=man, checkpoints_not_in_zip=ck, created=time.strftime("%Y-%m-%d %H:%M:%S"), profile=PROFILE), indent=1))
print(f"release pack: {REL} ({os.path.getsize(REL)/2**20:.1f} MB, {len(man)} files). Checkpoints ({len(ck)}) stay in {D['ckpt']}: upload them to Zenodo with the zip.")
print("Total time per stage:", {k: f"{v/60:.1f} min" for k, v in TIMERS.items()})

## 16. Optional: your own volumes

To apply the trained predictor, the conformal intervals, the out-of-distribution guard and the triage rule to new binary volumes (for example real micro-CT sub-volumes),
save them as `volumes` (uint8, shape N x 256 x 256 x 256, 1 = solid) in an `.npz`, optionally with `log_k` labels (log10 of permeability in px² of the same grid),
and set `EXTERNAL_NPZ` below. With labels, coverage on these volumes is reported; without labels, you get intervals, guard p-values and accept/simulate decisions.

In [ ]:
EXTERNAL_NPZ = ""
if EXTERNAL_NPZ:
    ext = np.load(EXTERNAL_NPZ); V = ext["volumes"]; assert V.shape[1] == FULL_RES, f"volumes must be {FULL_RES}^3"
    feats = []
    for s0 in range(0, len(V), 4):
        pore = torch.from_numpy(V[s0:s0 + 4] == 0).to(DEVICE); o, _ = featurise(pore, [K_W], cfg.poremap_cap); feats.append(o[K_W])
    Xe = torch.from_numpy(np.concatenate(feats)); ck = torch.load(os.path.join(D["ckpt"], "main_s0.pt"), map_location=DEVICE, weights_only=False)
    model = RockNet(2, P, N_Q, cfg.base, cfg.p_drop).to(DEVICE); model.load_state_dict(ck["model"]); model.eval()
    with torch.no_grad():
        x = torch.cat([Xe[:, 0:1].float() / FRAC_SCALE, Xe[:, 1:2].float() / PORE_SCALE], 1).to(DEVICE)
        mu, lv, q, z = model(x); ysn, ymn = ck["ys"].numpy(), ck["ym"].numpy()
    exd = dict(mu=mu.cpu().numpy() * ysn + ymn, sig=np.exp(0.5 * lv.cpu().numpy()) * ysn, q=q.cpu().numpy() * ysn[None, :, None] + ymn[None, :, None], feat=z.cpu().numpy(),
               y=np.full((len(V), P), np.nan))
    Pm = load_preds("main_s0"); lo, hi, _ = interval("cqr", Pm["cal"], exd, J_K, cfg.alpha, tr=Pm["tr"])
    f_tr = Pm["tr"]["feat"]; m_, s_ = f_tr.mean(0), f_tr.std(0) + 1e-6
    nov_e = knn_score((f_tr - m_) / s_, (exd["feat"] - m_) / s_); pv = conf_pvalue(NOV_MAIN["main_s0"]["cal"], nov_e)
    t = ltt_threshold(Pm["cal"]["sig"][:, J_K], np.abs(Pm["cal"]["mu"][:, J_K] - Pm["cal"]["y"][:, J_K]) > np.log10(F0), E0, cfg.ltt_delta, tri_grid(Pm["val"]["sig"][:, J_K]))
    out = pd.DataFrame(dict(pred_log_k=exd["mu"][:, J_K], lo_90=lo, hi_90=hi, guard_p=pv, flagged=pv <= cfg.guard_delta,
                            decision=np.where((exd["sig"][:, J_K] <= t) & (pv > cfg.guard_delta), "accept", "simulate")))
    if "log_k" in ext.files: out["log_k"] = ext["log_k"]; out["covered"] = (out.log_k >= lo) & (out.log_k <= hi)
    save_table(out, "T33_external_volumes")